# DC4 paper Figure 1 — NGC 4151 source-catalog fit

Original three-month full-mock source-catalog fit and comparison curves. Run cells in order. The isolated 24-month CPL+PL analysis is in Figure 2.


In [1]:
from pathlib import Path
from copy import deepcopy
import sys
import gc
import hashlib

import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import display

from astromodels import (
    Cutoff_powerlaw, ExtendedSource, PointSource, Parameter, Powerlaw, Model, load_model,
)
from histpy import Histogram
from threeML import DataList, JointLikelihood, BayesianAnalysis
from threeML.minimizer.minimization import FitFailed, CannotComputeCovariance
from threeML.exceptions.custom_exceptions import BadCovariance
from cosipy.threeml import BinnedSED
from dataclasses import dataclass
from scipy.optimize import brentq, minimize

from cosipy import BinnedData
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.data_io import EmCDSBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import (
    BinnedInstrumentResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
    BinnedThreeMLExtendedSourceResponse,
    ExtendedSourceResponse,
)
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.statistics import PoissonLikelihood
from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # noqa: F401


# DC4 recipe interpretation stays in this tutorial; reusable calculations live in cosipy.
import json
import yaml
from astropy.io import fits
from cosipy.response.temporal_profile import LightcurveProfile
from cosipy.response.threeml_temporal_response import (
    ZeroPointSourceResponse, ComponentPointSourceResponse,
    make_weighted_point_source_response,
)
from cosipy.util.megalib import megalib_map_to_galprop_fits

from threeML.plugin_prototype import PluginPrototype
from astromodels import Uniform_prior


## Shared catalog, spatial-template, and histogram helpers


In [3]:
def load_recipe(path, catalog_path, model):
    # A relocated recipe may be supplied in memory; retain the same validation.
    recipe = deepcopy(path) if isinstance(path, dict) else json.loads(Path(path).read_text())
    if recipe.get('schema') != 'dc4-component-lightcurve-response-v1':
        raise ValueError("Unsupported temporal-response recipe schema")
    if Path(recipe['catalog']).resolve() != Path(catalog_path).resolve():
        raise ValueError("The response recipe belongs to a different catalog")
    if recipe['selection'] not in {'full', 'NGC4151_GTI'} or recipe['time_basis'] != 'elapsed':
        raise ValueError("Expected a full-observation or NGC4151 GTI elapsed-time catalog")
    if set(row['source'] for row in recipe['components']) != set(model.sources):
        raise ValueError("Recipe and catalog source lists differ")
    if any(row['selection'] != recipe['selection'] for row in recipe['components']):
        raise ValueError("Mixed selections in response recipe")
    keys = [(r['source'], r['component']) for r in recipe['components']]
    if len(set(keys)) != len(keys):
        raise ValueError("Duplicate source/component recipe entries")
    return recipe


def component_name(source, label):
    if label in source.components:
        return label
    # v1 recipes store MEGAlib labels rather than astromodels component names.
    crab_names = {'Crab_Nebula': 'nebula', 'Crab_P1': 'peak1',
                  'Crab_Brg': 'bridge', 'Crab_P2': 'peak2'}
    if source.name == 'crab' and crab_names.get(label) in source.components:
        return crab_names[label]
    raise ValueError(f"Unknown temporal component {source.name}/{label}")


def make_temporal_responses(model, recipe, base_history, gti, data,
                            instrument_response, detector_response,
                            exposure_multiplier=1., full_history=None):
    """Build per-source/component responses from the unscaled GTI history.

    Validate temporal factors over the CATALOG selection, which can differ
    from the analysis GTI. Full catalogs require the original full_history:
    response weights remain L(t)/mean_full while exposure is GTI restricted.
    Exposure repetitions multiply response livetimes, not fluxes.
    Responses are computed lazily by cosipy and reused throughout fitting.
    """
    if not np.isfinite(exposure_multiplier) or exposure_multiplier <= 0:
        raise ValueError("Exposure multiplier must be finite and positive")
    starts, stops = np.asarray(gti.tstart_list.unix), np.asarray(gti.tstop_list.unix)
    duration = float(np.sum(stops-starts))
    if duration <= 0 or np.any(stops <= starts):
        raise ValueError("The analysis GTI must have positive duration")
    catalog_selection = recipe.get('selection', 'NGC4151_GTI')
    if catalog_selection == 'full':
        if full_history is None:
            raise ValueError("Full-observation catalogs require the original full_history")
        reference_starts = np.array([full_history.obstime[0].unix])
        reference_stops = np.array([full_history.obstime[-1].unix])
        if np.any(starts < reference_starts[0]-1e-5) or np.any(stops > reference_stops[0]+1e-5):
            raise ValueError("The analysis GTI extends beyond the catalog observation")
    elif catalog_selection == 'NGC4151_GTI':
        reference_starts, reference_stops = starts, stops
    else:
        raise ValueError(f"Unsupported catalog selection: {catalog_selection}")
    reference_duration = float(np.sum(reference_stops-reference_starts))
    if not np.isfinite(reference_duration) or reference_duration <= 0:
        raise ValueError("Invalid catalog reference duration")
    history_starts, history_stops = base_history.intervals_tstart.unix, base_history.intervals_tstop.unix
    rows_by_source = {}
    for row in recipe['components']:
        rows_by_source.setdefault(row['source'], []).append(row)
    overrides, diagnostics = {}, []
    for key, rows in rows_by_source.items():
        source = model.sources[key]
        for row in rows:
            if not np.isclose(row['selected seconds'], reference_duration, rtol=0., atol=1e-4):
                raise ValueError(f"Reference duration differs from the catalog: {key}")
        if not any(row['lightcurve'] for row in rows):
            continue
        if not isinstance(source, PointSource):
            raise NotImplementedError(f"A temporal extended response is required for {key}")
        if any(r['component'] == 'all' for r in rows) and len(rows) != 1:
            raise ValueError(f"Invalid component grouping for {key}")
        component_responses = {}
        for row in rows:
            applied = float(row['applied factor'])
            if not np.isfinite(applied) or applied <= 0:
                raise ValueError(f"Invalid applied factor for {key}")
            if row['lightcurve']:
                profile = LightcurveProfile.from_dat(
                    row['lightcurve'], expected_sha256=row['lightcurve sha256'],
                    repeating=row['repeating'], mode=row['normalization mode'],
                    bins=int(row['simulation bins']),
                )
                mean = float(profile.integral(reference_starts, reference_stops).sum()/reference_duration)
                gti_mean = float(profile.integral(starts, stops).sum()/duration)
                weights = profile.response_weights(history_starts, history_stops, applied)
                del profile
            else:
                mean, gti_mean, weights = 1., 1., np.ones(base_history.nintervals)
            if not np.isclose(mean, row['time factor'], rtol=1e-8, atol=1e-14):
                raise ValueError(f"Temporal factor differs from catalog for {key}/{row['component']}")
            if not np.isclose(applied, max(mean, 1e-30), rtol=1e-8, atol=0.):
                raise ValueError(f"Applied factor differs from catalog for {key}")
            weighted_live = base_history.livetime * weights * exposure_multiplier
            response = make_weighted_point_source_response(
                base_history, weights, data, instrument_response, detector_response,
                exposure_multiplier=exposure_multiplier,
            )
            diagnostics.append({'source': key, 'component': row['component'],
                                'normalization mode': row['normalization mode'],
                                'simulation bins': row['simulation bins'],
                                'catalog selection': catalog_selection,
                                'catalog time factor': mean,
                                'analysis GTI time factor': gti_mean,
                                'response denominator': applied,
                                'weighted response seconds': float(weighted_live.to_value(u.s).sum())})
            if row['component'] == 'all':
                overrides[key] = response
            else:
                name = component_name(source, row['component'])
                if name in component_responses:
                    raise ValueError(f"Duplicate mapped component for {key}/{name}")
                component_responses[name] = response
        if component_responses:
            if set(component_responses) != set(source.components):
                raise ValueError(f"Recipe does not cover all components of {key}")
            overrides[key] = ComponentPointSourceResponse(component_responses)
    return overrides, diagnostics


In [4]:
SPATIAL_TEMPLATE_NSIDE = 8
SPATIAL_TEMPLATE_FORMAT_TAG = "perMeV_v3"
SPATIAL_DEFINITIONS = {'positrons_26al_line': ('DC3', 'Positrons_from_26Al_line'),
 'positrons_26al_cont': ('DC3', 'Positrons_from_26Al_cont'),
 'positrons_44ti_line': ('DC3', 'Positrons_from_44Ti_line'),
 'positrons_44ti_cont': ('DC3', 'Positrons_from_44Ti_cont'),
 'narrow_bulge_511': ('DC3', 'NB_511'),
 'broad_bulge_511': ('DC3', 'BB_511'),
 'vela_snr_511': ('DC3', 'Vela_SNR_511'),
 'al26_ne2001': ('DC3', '26Al_NE2001'),
 'fe60_ne2001': ('DC3', '60Fe_NE2001'),
 'positrons_thin_disk_cont': ('DC4', 'Positrons_Thin_Disk_cont'),
 'positrons_thin_disk_line': ('DC4', 'Positrons_Thin_Disk_line')}

def resolve_source_asset(source_definition, filename):
    path = source_definition.parent / filename
    candidates = [path, Path(f"{path}.gz")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not resolve {filename!r} next to {source_definition}"
    )


def parse_megalib_blocks(source_definition):
    # Return one property dictionary per DataChallenge.Source block.
    blocks = []
    by_megalib_name = {}
    for raw_line in source_definition.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        tokens = line.split()
        if tokens[:1] == ["DataChallenge.Source"]:
            megalib_name = tokens[1]
            block = {"_name": megalib_name}
            blocks.append(block)
            by_megalib_name[megalib_name] = block
            continue
        if "." not in tokens[0]:
            continue
        megalib_name, prop = tokens[0].split(".", 1)
        if megalib_name in by_megalib_name:
            by_megalib_name[megalib_name][prop] = tokens[1:]
    if not blocks:
        raise ValueError(f"No DataChallenge.Source blocks in {source_definition}")
    return blocks


def source_library_path(value, source_library):
    """Relocate a catalog asset without relying on the original user's home."""
    parts = Path(value).parts
    if "Source_Library" not in parts:
        raise ValueError(f"Not a source-library path: {value}")
    relative = Path(*parts[parts.index("Source_Library") + 1:])
    candidate = (Path(source_library) / relative).resolve()
    if not candidate.is_file():
        raise FileNotFoundError(candidate)
    return str(candidate)


def ensure_spatial_template(source_name, output_path, source_library, overwrite=False):
    """Build only missing templates, preserving the generator's conversion."""
    output_path = Path(output_path)
    if output_path.exists() and not overwrite:
        with fits.open(output_path) as hdul:
            if (hdul['SKYMAP'].header.get('NSIDE') != SPATIAL_TEMPLATE_NSIDE
                    or hdul['SKYMAP'].header.get('MAPVERS') != SPATIAL_TEMPLATE_FORMAT_TAG):
                raise ValueError(f"Incompatible cached spatial template: {output_path}")
        return output_path, 'loaded'
    challenge, stem = SPATIAL_DEFINITIONS[source_name]
    matches = sorted((Path(source_library) / challenge / 'sources').rglob(f'{stem}.source'))
    if len(matches) != 1:
        raise ValueError(f"Expected one {stem}.source, found {matches}")
    blocks = parse_megalib_blocks(matches[0])
    if len(blocks) != 1 or len(blocks[0].get('Beam', [])) < 2:
        raise ValueError(f"Expected one coupled map in {matches[0]}")
    map_path = resolve_source_asset(matches[0], blocks[0]['Beam'][1])
    megalib_map_to_galprop_fits(
        map_path, output_path, nside=SPATIAL_TEMPLATE_NSIDE,
        oversample_nside=64, energy_bounds=(100., 10000.), overwrite=overwrite,
    )
    return output_path, 'created'


def prepare_catalog(catalog_path, recipe_input, source_library, template_directory,
                    orientation_path, response_path, binning_path, overwrite=False):
    """Generate templates, relocate external assets, and load without rewriting YAML.

    The YAML and base JSON remain byte-for-byte unchanged. Only
    the in-memory dictionaries receive this computer's paths. Their source
    amplitudes and the recipe's light-curve hashes are never modified.
    """
    from astromodels.core.model_parser import ModelParser
    from cosipy.threeml.custom_functions import GalpropHealpixModel, SpecFromDat  # noqa: F401

    catalog_path = Path(catalog_path)
    contents = yaml.safe_load(catalog_path.read_text())
    recipe = deepcopy(recipe_input)
    if Path(recipe['catalog']).name != catalog_path.name:
        raise ValueError('The recipe refers to a different catalog filename')
    recipe['catalog'] = str(catalog_path.resolve())
    for key, local in [('orientation', orientation_path), ('detector_response', response_path),
                       ('binning', binning_path)]:
        if Path(recipe[key]).name != Path(local).name:
            raise ValueError(f'{key} filename differs from the catalog recipe')
        recipe[key] = str(Path(local).resolve())
    for row in recipe['components']:
        if row['lightcurve']:
            row['lightcurve'] = source_library_path(row['lightcurve'], source_library)

    template_rows = []
    def relocate(node, source_name):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == '_fitsfile' and value:
                    expected_name = f'{source_name}_nside8_perMeV_v3.fits'
                    if Path(value).name != expected_name:
                        raise ValueError(f'Unsupported spatial template: {value}')
                    path, status = ensure_spatial_template(
                        source_name, Path(template_directory) / expected_name,
                        source_library, overwrite=overwrite,
                    )
                    node[key] = str(path.resolve())
                    template_rows.append({'source': source_name, 'status': status, 'file': str(path)})
                elif isinstance(value, str) and 'Source_Library/' in value:
                    node[key] = source_library_path(value, source_library)
                else:
                    relocate(value, source_name)
        elif isinstance(node, list):
            for value in node:
                relocate(value, source_name)
    for source_key, definition in contents.items():
        relocate(definition, source_key.split(' (')[0])
    model = ModelParser(model_dict=contents).get_model()
    recipe = load_recipe(recipe, catalog_path, model)
    return model, recipe, template_rows


In [5]:
def load_selected_histogram(path, gti, binning_path, already_selected=True):
    """Load explicitly preselected HDF5 or select native FITS events before binning.

    Histogram axes do not encode their GTI. For an already-selected histogram,
    its time-selection provenance is the caller's responsibility. An unselected
    projected histogram is rejected because event times cannot be recovered.
    Native event selection uses half-open [start, stop) GTI intervals.
    """
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    from histpy import Histogram, Axis, Axes, HealpixAxis

    config = yaml.safe_load(Path(binning_path).read_text())
    axes = Axes([
        Axis(config['energy_bins'], unit=u.keV, label='Em'),
        Axis(np.linspace(0., 180., int(180. / config['phi_pix_size']) + 1),
             unit=u.deg, label='Phi'),
        HealpixAxis(nside=config['nside'], scheme=config['scheme'],
                    coordsys='galactic', label='PsiChi'),
    ])
    path = Path(path)
    is_fits = path.name.endswith(('.fits', '.fits.gz'))
    if not is_fits:
        if not already_selected:
            raise ValueError('Use native event FITS to apply a new GTI, not a projected histogram')
        histogram = Histogram.open(path).project('Em', 'Phi', 'PsiChi')
        if histogram.axes != axes:
            raise ValueError(f'Histogram binning differs from agn.yaml: {path}')
        return histogram

    starts = np.asarray(gti.tstart_list.unix, dtype=float)
    stops = np.asarray(gti.tstop_list.unix, dtype=float)
    if (starts.size == 0 or np.any(stops <= starts)
            or np.any(starts[1:] < stops[:-1])):
        raise ValueError('GTI intervals must be positive, sorted and nonoverlapping')
    histogram = Histogram(axes, sparse=True)
    with fits.open(path, memmap=False) as hdul:
        events = hdul[1].data
        for offset in range(0, len(events), 250_000):
            block = events[offset:offset + 250_000]
            times = np.asarray(block['TimeTags'], dtype=float)
            if already_selected:
                selected = np.ones(times.size, dtype=bool)
            else:
                index = np.searchsorted(starts, times, side='right') - 1
                selected = (index >= 0) & (times < stops[np.maximum(index, 0)])
            direction = SkyCoord(
                l=np.asarray(block['Chi galactic'][selected]) * u.deg,
                b=np.asarray(block['Psi galactic'][selected]) * u.deg,
                frame='galactic',
            )
            histogram.fill(np.asarray(block['Energies'][selected]) * u.keV,
                           np.asarray(block['Phi'][selected]) * u.rad, direction)
    return histogram


## Input files and NGC 4151 GTI

The first histogram is the original mock supplied by the user. Figure 1 bins the original DC4 NGC 4151 event FITS under the same pointing GTI and adds the selected instrumental background. Figure 2 uses a source-only HDF5 with altered PL normalization specified as 0.30 times the CPL differential flux at 200 keV. The mock and modified source histograms were preselected to the NGC 4151 field of view; those binned files cannot be cut to another GTI. The original source FITS retains event times and is explicitly selected by this notebook. The first plot uses three-month data for all its fits. Only the second, isolated CPL + PL plot repeats the source and instrumental-background counts and the selected pointing livetime eight times to forecast 24 months.

For provenance: the earlier altered full mock has 21,189 more reconstructed counts than the unedited mock; 17,501 of those additional counts are in the first two `Em` bins (100–251 keV). The two catalog fits therefore use different target signals. This count comparison alone does not establish that either CPL optimizer found the global optimum.


In [7]:
# Locate the AGN tutorial from its own folder or the cosipy checkout.
notebook_dir = Path.cwd().resolve()
if not (notebook_dir / "agn_paper_plot_style.py").is_file():
    roots = [p for p in (notebook_dir, *notebook_dir.parents) if (p / "cosipy").is_dir()]
    if not roots:
        raise FileNotFoundError("Run from this tutorial folder or the cosipy checkout")
    notebook_dir = roots[0] / "docs/tutorials/spectral_fits/continuum_fit/AGN"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

dc4_candidates = [
    Path.home() / "Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files",
    Path.home() / "Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files",
]
dc4_files_root = next((p for p in dc4_candidates if p.is_dir()), None)
if dc4_files_root is None:
    raise FileNotFoundError(f"DC4 files not found in {dc4_candidates}")
source_library = Path.home() / "Software/cosi-sim/cosi_sim/Source_Library"
mock_data_path = dc4_files_root / "Mock_Data/Mock_Data_Cut/dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered_ngc4151_in_fov_binned.hdf5"
updated_mock_data_path = dc4_files_root / "Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p30_time_cut.hdf5"
# The isolated source-only histogram uses a 0.30 PL/CPL ratio at 200 keV.
ngc_original_fits_path = dc4_files_root / (
    "DC4_Sources/DC4/NGC_4151_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
)
ngc_only_source_path = dc4_files_root / (
    "DC4_Sources/DC4_Mod/NGC_4151_ec200_0p30_DC4_COSI_cpl_pl_time_cut_in_fov.hdf5"
)
background_path = dc4_files_root / (
    "Background/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_"
    "withSAAbck_NGC4151_60deg_fov_cut.hdf5"
)
orientation_path = dc4_files_root / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
response_path = dc4_files_root / (
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491."
    "m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)
agn_config = notebook_dir / "agn.yaml"
parallel_catalog_path = notebook_dir / "source_catalog_DC4_all_71files_NGC4151_GTI_elapsed_lightcurve_response_norm.yaml"
parallel_recipe_path = parallel_catalog_path.with_suffix(".response.json")
spatial_template_directory = dc4_files_root / "DC4_Sources/DC4_Spatial_Templates"
extended_response_path = dc4_files_root / (
    "Extended_Responses/NGC4151_Cut/"
    "DC4_continuum_NGC4151_60deg_GTI_extended_response_nside8.h5"
)
mock_already_gti_selected = True
background_already_gti_selected = True
exposure_multiplier = 1

for path in (source_library, mock_data_path, updated_mock_data_path,
             ngc_original_fits_path,
             ngc_only_source_path, background_path,
             orientation_path, response_path, agn_config, parallel_catalog_path,
             parallel_recipe_path, extended_response_path):
    if not path.exists():
        raise FileNotFoundError(path)
print("Original full mock:", mock_data_path)
print("Updated 0.30 full mock:", updated_mock_data_path)
print("Parallel catalog:", parallel_catalog_path)
print("Original NGC event FITS for Figure 1:", ngc_original_fits_path)
print("Modified NGC source for Figure 2:", ngc_only_source_path)
if exposure_multiplier <= 0:
    raise ValueError("exposure_multiplier must be positive")


Original full mock: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/Mock_Data/Mock_Data_Cut/dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered_ngc4151_in_fov_binned.hdf5
Updated 0.30 full mock: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p30_time_cut.hdf5
Parallel catalog: /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/source_catalog_DC4_all_71files_NGC4151_GTI_elapsed_lightcurve_response_norm.yaml
Original NGC event FITS for Figure 1: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/DC4_Sources/DC4/NGC_4151_3months_unbinned_data_filtered_with_SAAcut.fits.gz
Modified NGC source for Figure 2: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/DC4_Sources/DC4_Mod/NGC_4151_ec200_0p30_DC4_

In [8]:
parallel_recipe_input = json.loads(parallel_recipe_path.read_text())
parallel_model, parallel_recipe, template_status = prepare_catalog(
    parallel_catalog_path, parallel_recipe_input, source_library,
    spatial_template_directory, orientation_path, response_path, agn_config,
    overwrite=False,
)
method_models = {"parallel chunks": parallel_model}
method_recipes = {"parallel chunks": parallel_recipe}
if len(parallel_model.sources) != 71 or parallel_model.free_parameters:
    raise RuntimeError("Expected a frozen, 71-entry parallel catalog")
sc_orientation_full = SpacecraftHistory.open(orientation_path)
print("Catalog sources:", len(parallel_model.sources))
display(pd.DataFrame(template_status))


Catalog sources: 71
                      source  status  \
0        positrons_26al_line  loaded   
1        positrons_26al_cont  loaded   
2        positrons_44ti_line  loaded   
3        positrons_44ti_cont  loaded   
4           narrow_bulge_511  loaded   
5            broad_bulge_511  loaded   
6               vela_snr_511  loaded   
7                al26_ne2001  loaded   
8                fe60_ne2001  loaded   
9   positrons_thin_disk_cont  loaded   
10  positrons_thin_disk_line  loaded   

                                                 file  
0   /Users/parshadkp/Library/CloudStorage/OneDrive...  
1   /Users/parshadkp/Library/CloudStorage/OneDrive...  
2   /Users/parshadkp/Library/CloudStorage/OneDrive...  
3   /Users/parshadkp/Library/CloudStorage/OneDrive...  
4   /Users/parshadkp/Library/CloudStorage/OneDrive...  
5   /Users/parshadkp/Library/CloudStorage/OneDrive...  
6   /Users/parshadkp/Library/CloudStorage/OneDrive...  
7   /Users/parshadkp/Library/CloudStorage/OneDrive.

In [9]:
source_l = parallel_model.ngc4151.position.l.value * u.deg
source_b = parallel_model.ngc4151.position.b.value * u.deg
source_coord = SkyCoord(l=source_l, b=source_b, frame="galactic")

# These settings define the shared NGC 4151 GTI selection.
max_offaxis = 60.0 * u.deg
earth_occ = True

source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    sc_orientation_full,
    max_offaxis,
    earth_occ=earth_occ,
)
sc_orientation_cut = sc_orientation_full.apply_gti(source_gti)

# Repeat the selected pointing history by scaling its interval livetimes.
# The attitude and Earth-occultation selection remain unchanged.
sc_orientation = SpacecraftHistory(
    sc_orientation_cut.obstime,
    sc_orientation_cut.attitude,
    sc_orientation_cut.location,
    livetime=sc_orientation_cut.livetime * exposure_multiplier,
)
sc_orientation.cache_earth_occ = sc_orientation_cut.cache_earth_occ


full_livetime = sc_orientation_full.cumulative_livetime().to_value(u.s)
fov_livetime = sc_orientation_cut.cumulative_livetime().to_value(u.s)
scaled_livetime = sc_orientation.cumulative_livetime().to_value(u.s)
print(f"Full orientation livetime: {full_livetime:,.1f} s")
print(f"Three-month GTI livetime: {fov_livetime:,.1f} s")
print(f"Scaled {(3 * exposure_multiplier)}-month livetime: {scaled_livetime:,.1f} s")


Full orientation livetime: 6,579,555.0 s
Three-month GTI livetime: 980,415.0 s
Scaled 3-month livetime: 980,415.0 s


In [10]:
data_hist = load_selected_histogram(
    mock_data_path, source_gti, agn_config,
    already_selected=mock_already_gti_selected,
)
total_bkg = load_selected_histogram(
    background_path, source_gti, agn_config,
    already_selected=background_already_gti_selected,
)
ngc_original_source_hist = load_selected_histogram(
    ngc_original_fits_path, source_gti, agn_config, already_selected=False,
)
ngc_only_source_hist = load_selected_histogram(
    ngc_only_source_path, source_gti, agn_config, already_selected=True,
)
if not (total_bkg.axes == data_hist.axes == ngc_original_source_hist.axes
        == ngc_only_source_hist.axes):
    raise ValueError("Full mock, NGC-only, and background detector axes differ")
data_hist *= exposure_multiplier
total_bkg *= exposure_multiplier
ngc_original_source_hist *= exposure_multiplier
ngc_only_source_hist *= exposure_multiplier
total_bkg += sys.float_info.min
# Figure 1 uses the original DC4 NGC 4151 events selected by the GTI.
# Figure 2 uses only the altered 0.30 source; both add instrumental background.
# Neither isolated histogram includes the other 70 sky sources or full mock.
comparison_data_hist_three_month = ngc_original_source_hist + total_bkg
one_source_data_hist_three_month = ngc_only_source_hist + total_bkg
print("Original NGC GTI event counts:",
      float(ngc_original_source_hist.to_dense(copy=False).contents.sum()))
print("Figure 1 original source + background counts:",
      float(comparison_data_hist_three_month.to_dense(copy=False).contents.sum()))
print("Figure 2 modified 0.30 source + background counts (three-month base):",
      float(one_source_data_hist_three_month.to_dense(copy=False).contents.sum()))
bkg_dist = {"total_bkg": total_bkg}
background_rate_initial = float(total_bkg.to_dense(copy=False).contents.sum()) / scaled_livetime
if not np.isfinite(background_rate_initial) or background_rate_initial <= 0:
    raise ValueError("Invalid initial instrumental-background rate")
print(f"Initial instrumental-background rate: {background_rate_initial:.6g} Hz")
dr = FullDetectorResponse.open(str(response_path))


Original NGC GTI event counts: 54915.0
Figure 1 original source + background counts: 25125949.0
Figure 2 modified 0.30 source + background counts (three-month base): 25147137.398761287
Initial instrumental-background rate: 25.5719 Hz


In [11]:
# Construct the ordinary responses once; they do not depend on LC normalization.
unpolarized_components = []
for model in (parallel_model,):
    for source in model.sources.values():
        for component in source.components.values():
            degree = getattr(component.polarization, "degree", None)
            if degree is not None and degree.value != 0.0:
                degree.value = 0.0
                unpolarized_components.append(f"{source.name}.{component.name}")
print("Using unpolarized folding for:", sorted(set(unpolarized_components)))

data = EmCDSBinnedData(data_hist)
instrument_response = BinnedInstrumentResponse(dr, data)
psr = BinnedThreeMLPointSourceResponse(
    data=data, instrument_response=instrument_response, sc_history=sc_orientation,
    energy_axis=dr.axes["Ei"],
    polarization_axis=dr.axes["Pol"] if "Pol" in dr.axes.labels else None,
    nside=2 * data.axes["PsiChi"].nside,
)
if extended_response_path.exists():
    print("Loading cached GTI extended response (about 13 GiB)...")
    extended_response = ExtendedSourceResponse.open(extended_response_path)
else:
    print("Generating the NGC 4151 GTI extended response; this can be slow...")
    extended_response = dr.get_extended_source_response(
        sc_orientation_cut, coordsys="galactic", nside_image=8,
        nside_scatt_map=2 * data.axes["PsiChi"].nside, earth_occ=earth_occ,
    )
    extended_response_path.parent.mkdir(parents=True, exist_ok=True)
    extended_response.write(extended_response_path, overwrite=False)
if extended_response.axes["NuLambda"].nside != 8:
    raise ValueError("Expected an NSIDE-8 extended response")
if extended_response.axes[2:] != data.axes:
    raise ValueError("Extended response and data axes differ")
if extended_response.axes["Ei"] != dr.axes["Ei"]:
    raise ValueError("Point and extended incident-energy axes differ")
extended_response *= exposure_multiplier
esr = BinnedThreeMLExtendedSourceResponse(data=data, precomputed_psr=extended_response)
print("Shared ordinary responses are ready.")


Using unpolarized folding for: ['crab.bridge', 'crab.nebula', 'crab.peak1', 'crab.peak2', 'grb_bn080802386_flux150.bn080802386', 'grb_bn090424592.bn090424592', 'grb_bn100612726.bn100612726', 'grb_bn110605183.bn110605183', 'grb_bn131122490.bn131122490', 'grb_bn140329295.bn140329295', 'grb_bn161004964.bn161004964', 'grb_bn170405777.bn170405777', 'grb_bn180504136.bn180504136', 'grb_bn180703876.bn180703876', 'mgf_051103.mgf051103', 'mgf_070201.mgf070201', 'mgf_070222.mgf070222', 'mgf_180128a.mgf180128a', 'mgf_200415a.mgf200415a', 'mgf_231115a.mgf231115a']
Loading cached GTI extended response (about 13 GiB)...
Shared ordinary responses are ready.


## Injected references and fit helpers

The original DC4 source library supplies the NGC 4151 injected spectrum. For the catalog plot, read its tabulated `.dat` spectrum directly instead of using the altered 0.30 reference. The isolated plot uses the 0.30 CPL + PL model. The catalog fit itself replaces NGC 4151 with a free CPL; the other 70 sources remain fixed.


In [13]:
# Original unedited DC4 injection is tabulated in the source library.
original_ngc_spectrum_path = (source_library / "DC4/sources/Extragalactic/NGC_4151/"
                              "NGC4151_ec200_TH_and_NTH_COSI.dat")
if not original_ngc_spectrum_path.exists():
    raise FileNotFoundError(original_ngc_spectrum_path)
# The MEGAlib file also contains an "IP LINLIN" interpolation directive.
with original_ngc_spectrum_path.open() as spectrum_file:
    original_spectrum = np.array([
        [float(fields[1]), float(fields[2])]
        for line in spectrum_file
        if (fields := line.split()) and fields[0] == "DP"
    ], dtype=float)
original_energy_keV, original_photon_flux = original_spectrum.T
if len(original_energy_keV) < 2 or np.any(np.diff(original_energy_keV) <= 0):
    raise ValueError("Original NGC 4151 spectrum energy samples are not increasing")

def original_injected_flux(energies_keV):
    # MEGAlib's .dat definition is linearly interpolated in energy and flux.
    return np.interp(energies_keV, original_energy_keV, original_photon_flux,
                     left=np.nan, right=np.nan)

# The isolated data file was generated with a 0.30 PL/CPL ratio at 200 keV.
injected_pl_to_cpl_at_200_keV = 0.30
spectrum_inj_cpl = Cutoff_powerlaw()
spectrum_inj_cpl.K.value = 0.15
spectrum_inj_cpl.piv.value = 1.0
spectrum_inj_cpl.xc.value = 200.0
spectrum_inj_cpl.index.value = -1.75
spectrum_inj_cpl.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum_inj_cpl.piv.unit = u.keV
spectrum_inj_cpl.xc.unit = u.keV
spectrum_inj_tail = Powerlaw()
spectrum_inj_tail.K.value = injected_pl_to_cpl_at_200_keV * spectrum_inj_cpl.evaluate_at(200.0)
spectrum_inj_tail.piv.value = 200.0
spectrum_inj_tail.index.value = -3.8
spectrum_inj_tail.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum_inj_tail.piv.unit = u.keV
spectrum_inj_isolated = spectrum_inj_cpl + spectrum_inj_tail
energy = np.geomspace(100 * u.keV, 10 * u.MeV).to_value(u.keV)
injected_flux = original_injected_flux(energy)
binned_energy_edges = data_hist.axes["Em"].edges.to_value(u.keV)
binned_energy = 0.5 * (binned_energy_edges[1:] + binned_energy_edges[:-1])
observed_counts = np.asarray(data_hist.project("Em").to_dense(copy=False).contents, dtype=float).copy()


# The original tabulated spectrum contains both thermal and non-thermal flux.
# The generator's CPL component is shown separately to judge the isolated
# CPL-only comparison at low energy.
def injected_thermal_flux(energies_keV):
    k200 = 0.15 * 200.0**(-1.75)
    e = np.asarray(energies_keV, dtype=float)
    return k200 * (e / 200.0)**(-1.75) * np.exp(-e / 200.0)

def make_fit_model(model):
    """Replace only NGC 4151's fixed injection spectrum with a free CPL."""
    assert not model.free_parameters
    source = model.ngc4151
    shape = source.spectrum.main.composite
    cpl = Cutoff_powerlaw(
        K=shape.K_1.value, piv=shape.piv_1.value,
        index=shape.index_1.value, xc=shape.xc_1.value,
    )
    fitted_source = PointSource(
        "ngc4151", l=source.position.l.value, b=source.position.b.value,
        spectral_shape=cpl,
    )
    for parameter in fitted_source.parameters.values():
        parameter.fix = True
    model.remove_source("ngc4151")
    model.add_source(fitted_source)
    shape = model.ngc4151.spectrum.main.Cutoff_powerlaw
    shape.K.min_value, shape.K.max_value = 1e-8, 1e-2
    shape.K.delta, shape.K.fix = 0.1 * shape.K.value, False
    # astromodels uses a signed exponent: photon index Γ>0 means index<0.
    # Keep the injected index -1.75 inside the allowed scan.
    shape.index.min_value, shape.index.max_value = -5., -0.01
    shape.index.delta, shape.index.fix = 0.1, False
    shape.xc.min_value, shape.xc.max_value = 100., 10000.
    shape.xc.delta, shape.xc.fix = 0.1 * shape.xc.value, False
    shape.piv.fix = True
    expected = {
        "ngc4151.spectrum.main.Cutoff_powerlaw.K",
        "ngc4151.spectrum.main.Cutoff_powerlaw.index",
        "ngc4151.spectrum.main.Cutoff_powerlaw.xc",
    }
    if set(model.free_parameters) != expected:
        raise ValueError("Catalog CPL fit must free only NGC 4151 K, index and xc")
    return model


In [14]:
def fit_joint_with_fallback(joint, label):
    """Fit the model and explicitly reject missing or invalid covariance."""
    initial_values = {name: par.value for name, par in joint._free_parameters.items()}
    try:
        joint.fit(quiet=True)
    except (FitFailed, CannotComputeCovariance, BadCovariance) as exc:
        print(f"{label}: MINUIT {type(exc).__name__}; retrying with SCIPY")
        # A failed MIGRAD may leave astromodels at its last trial point.
        # A covariance failure happens after a valid minimum, so retain it.
        if isinstance(exc, FitFailed):
            for name, par in joint._free_parameters.items():
                par.value = initial_values[name]
        joint.set_minimizer("SCIPY")
        try:
            joint.fit(quiet=True)
        except (CannotComputeCovariance, BadCovariance):
            joint.fit(quiet=True, compute_covariance=False)
    joint.restore_best_fit()
    covariance = np.atleast_2d(np.asarray(joint.results.covariance_matrix, dtype=float))
    covariance_valid = (
        np.all(np.isfinite(covariance))
        and np.all(np.diag(covariance) > 0)
    )
    if covariance_valid:
        try:
            np.linalg.cholesky(covariance)
        except np.linalg.LinAlgError:
            covariance_valid = False
    if not covariance_valid:
        print(f"{label}: no valid covariance; global 68% band is unavailable. "
              "A threeML '+/- 0' after compute_covariance=False is only a placeholder.")
    return covariance_valid


def parameter_intervals_68(results, parameters, covariance_valid):
    """Return valid covariance-based 68% intervals for fitted parameters.

    threeML's equal-tail intervals use random draws from the covariance and
    reject samples outside parameter bounds. When that truncation shifts the
    equal-tail interval away from the MLE, use the local 1-sigma covariance
    interval in the parameter's internal (possibly log) coordinates instead.
    """
    if not covariance_valid:
        return None
    covariance = np.atleast_2d(np.asarray(results.covariance_matrix, dtype=float))
    free = results.optimized_model.free_parameters
    names = list(free)
    intervals = {}
    for label, parameter in parameters.items():
        if parameter.path not in free:
            continue
        fitted_parameter = free[parameter.path]
        value = float(fitted_parameter.value)
        # A Hessian interval at a hard boundary can look spuriously tiny.
        # Report no two-sided 68% interval for a pegged parameter; use a
        # profile likelihood or a separately stated fixed-index fit instead.
        lower_bound = fitted_parameter.min_value
        upper_bound = fitted_parameter.max_value
        if lower_bound is not None and upper_bound is not None:
            bound_tolerance = 1e-5 * (float(upper_bound) - float(lower_bound))
        else:
            bound_tolerance = 1e-8 * max(abs(value), 1e-12)
        if ((lower_bound is not None and value - float(lower_bound) <= bound_tolerance)
                or (upper_bound is not None and float(upper_bound) - value <= bound_tolerance)):
            print(f"{label}: parameter is at a bound; two-sided 68% interval unavailable")
            continue
        try:
            lower, upper = map(float,
                               results.get_variates(parameter.path).equal_tail_interval(cl=0.68))
            if not (np.isfinite(lower) and np.isfinite(upper)
                    and lower <= value <= upper and lower < upper):
                raise ValueError("Bound-truncated samples do not enclose the MLE")
        except (AssertionError, BadCovariance, CannotComputeCovariance, ValueError):
            i = names.index(parameter.path)
            sigma_internal = float(np.sqrt(covariance[i, i]))
            center_internal = float(fitted_parameter._get_internal_value())
            if fitted_parameter.has_transformation():
                transform = fitted_parameter.transformation
                lower = float(transform.backward(center_internal - sigma_internal))
                upper = float(transform.backward(center_internal + sigma_internal))
            else:
                lower = center_internal - sigma_internal
                upper = center_internal + sigma_internal
            if fitted_parameter.min_value is not None:
                lower = max(lower, float(fitted_parameter.min_value))
            if fitted_parameter.max_value is not None:
                upper = min(upper, float(fitted_parameter.max_value))
            print(f"{label}: using local-covariance 68% interval because "
                  "threeML's bound-truncated equal-tail interval excludes the best fit")
        if (np.isfinite(lower) and np.isfinite(upper)
                and lower <= value <= upper and lower < upper):
            intervals[label] = [lower, upper]
        else:
            print(f"{label}: 68% interval unavailable after covariance check")
    return intervals or None


def format_parameter_68(record, label, spec):
    """Format a fitted value with asymmetric 68% errors, if they were saved."""
    value = float(record[label])
    interval = (record.get("parameter intervals 68") or {}).get(label)
    if interval is None:
        return f"{value:{spec}} (68% unavailable)"
    lower, upper = map(float, interval)
    if not (np.isfinite(lower) and np.isfinite(upper)
            and lower <= value <= upper and lower < upper):
        return f"{value:{spec}} (68% unavailable)"
    return (f"{value:{spec}} "
            f"(−{value - lower:{spec}}/+{upper - value:{spec}})")


def refresh_dc4_response_integrators():
    """Use on-disk spectral and diffuse integration code in a live kernel."""
    import importlib
    import cosipy.response.integrals as response_integrals
    import cosipy.response.functions_3d as functions_3d

    importlib.reload(response_integrals)
    importlib.reload(functions_3d)
    extended_module = importlib.import_module("cosipy.response.ExtendedSourceResponse")
    extended_module.get_integrated_extended_model_3d = (
        functions_3d.get_integrated_extended_model_3d
    )


## Independent SED likelihood and Bayesian weak-bin limits

Each SED bin uses BinnedSED’s fixed E^-2 power law, folded through the same source response. Other catalog sources and the fitted instrumental background remain fixed. Detections and TS ≥ 1 upper limits use the existing profile likelihood. For each TS < 1 bin, threeML/emcee samples its nonnegative amplitude with a uniform-flux prior while the other SED bins remain at their joint MLEs. These are conditional 95% credible upper limits; the prior bounds and chain diagnostics are recorded with the result.


In [16]:
@dataclass
class JointSEDFitResult:
    k: np.ndarray
    scaled_k: np.ndarray
    relative_nll: float
    ts: np.ndarray | None = None
    lower68: np.ndarray | None = None
    upper68: np.ndarray | None = None
    upper95: np.ndarray | None = None


class JointSEDTemplates:
    def __init__(self, background, probe_templates, probe_k, max_k):
        self.background = np.asarray(background, dtype=float).ravel()
        templates = np.asarray(probe_templates, dtype=float)
        self.probe_k = np.asarray(probe_k, dtype=float)
        self.max_k = np.asarray(max_k, dtype=float)
        if templates.ndim < 2 or templates.shape[0] != len(self.probe_k):
            raise ValueError("One detector template is required for each SED bin.")
        self.templates = templates.reshape(len(self.probe_k), -1)
        if self.templates.shape[1] != self.background.size:
            raise ValueError("Templates and background have different detector cells.")
        invalid = []
        for name, counts in (("fixed sky + instrumental background", self.background),
                             ("NGC 4151 SED templates", self.templates)):
            n_nonfinite = int(np.count_nonzero(~np.isfinite(counts)))
            n_negative = int(np.count_nonzero(counts < 0))
            if n_nonfinite or n_negative:
                invalid.append(f"{name}: {n_nonfinite} nonfinite, {n_negative} negative")
        if invalid:
            raise ValueError("Invalid Poisson counts: " + "; ".join(invalid))
        if np.any(self.probe_k <= 0) or np.any(self.max_k <= 0):
            raise ValueError("Probe and maximum normalizations must be positive.")
        self.probe_totals = self.templates.sum(axis=1)
        self.active_bins = np.flatnonzero(self.probe_totals > 0)
        if self.active_bins.size == 0:
            raise ValueError("No SED bin has a nonzero detector response.")
        self.k_scale = np.zeros(len(self.probe_k))
        self.k_scale[self.active_bins] = np.minimum(
            self.max_k[self.active_bins],
            self.probe_k[self.active_bins] * 10.0 / self.probe_totals[self.active_bins],
        )
        self.count_templates = self.templates[self.active_bins] * (
            self.k_scale[self.active_bins] / self.probe_k[self.active_bins]
        )[:, None]
        self.count_totals = self.count_templates.sum(axis=1)
        self.upper_scaled = self.max_k[self.active_bins] / self.k_scale[self.active_bins]

    def fit(self, data, initial_k=None, profile_ts=False, intervals=False):
        """Fit all K_i at once; optionally profile each bin's TS and limits."""
        data = np.asarray(data, dtype=float).ravel()
        if data.shape != self.background.shape or np.any(data < 0):
            raise ValueError("Data and background must match and be non-negative.")
        # Occupied cells without any SED response add only a constant to the
        # likelihood. Excluding them also avoids huge ratios from tiny empty
        # background-template cells.
        responsive = np.any(self.count_templates > 0, axis=0)
        occupied = (data > 0) & responsive
        background = self.background[occupied]
        if np.any(background <= 0):
            raise ValueError("Background must be positive in responsive data cells.")
        observed = data[occupied]
        templates = self.count_templates[:, occupied]
        totals = self.count_totals
        n_free = len(self.active_bins)
        # Keep free amplitudes just above zero during optimization. The null
        # value remains exactly zero for TS profiling. With a subnormal
        # instrumental-background floor, an optimizer trial at x=0 can give
        # a ~1e242 score and make L-BFGS-B report ABNORMAL.
        lower_scaled = np.minimum(1e-8, 1e-3 * self.upper_scaled)
        bounds = list(zip(lower_scaled, self.upper_scaled))
        if initial_k is None:
            x0 = np.ones(n_free)
        else:
            initial_k = np.asarray(initial_k, dtype=float)
            x0 = initial_k[self.active_bins] / self.k_scale[self.active_bins]
            x0 = np.clip(x0, lower_scaled, self.upper_scaled)
        # The probe scaling targets roughly ten counts per active bin. Avoid
        # starting all bins at zero when the preceding global CPL fit is weak.
        x0 = np.maximum(x0, np.minimum(1.0, self.upper_scaled))
        reference = background + x0 @ templates
        if np.any(reference <= 0) or np.any(~np.isfinite(reference)):
            raise ValueError("The initial SED model has invalid Poisson expectations.")
        log_reference = np.log(reference)

        def objective(x):
            source = x @ templates
            expected = background + source
            if np.any(expected <= 0) or np.any(~np.isfinite(expected)):
                return np.inf, np.full(n_free, np.nan)
            # log(expected)-log(reference) stays finite when the background
            # is subnormal. log1p(source/background) can overflow to infinity.
            value = float(totals @ x - observed @ (np.log(expected) - log_reference))
            ratio = observed / np.maximum(expected, 1e-250)
            gradient = totals - templates @ ratio
            return value, gradient

        optimum = minimize(
            objective, x0, method="L-BFGS-B", jac=True, bounds=bounds,
            options={"maxiter": 2000, "ftol": 1e-12, "gtol": 1e-6},
        )
        if not optimum.success or not np.isfinite(optimum.fun):
            retry_start = np.minimum(np.ones(n_free), self.upper_scaled)
            optimum = minimize(
                objective, retry_start, method="L-BFGS-B", jac=True, bounds=bounds,
                options={"maxiter": 5000, "maxls": 80, "ftol": 1e-12, "gtol": 1e-7},
            )
        if not optimum.success or not np.isfinite(optimum.fun):
            raise RuntimeError(
                f"Joint BinnedSED fit failed: {optimum.message}; "
                f"active bins={self.active_bins.tolist()}, "
                f"probe totals={self.probe_totals.tolist()}"
            )
        best = np.asarray(optimum.x, dtype=float)
        best_nll = float(objective(best)[0])
        k = np.zeros(len(self.probe_k))
        k[self.active_bins] = best * self.k_scale[self.active_bins]
        result = JointSEDFitResult(k=k, scaled_k=best, relative_nll=best_nll)
        if not profile_ts and not intervals:
            return result

        def profile_nll(local_index, fixed_value):
            if fixed_value == best[local_index]:
                return best_nll
            other = np.arange(n_free) != local_index
            trial = best.copy()
            trial[local_index] = fixed_value
            if not np.any(other):
                return float(objective(trial)[0])

            def other_objective(values):
                trial[other] = values
                value, gradient = objective(trial)
                return value, gradient[other]

            profiled = minimize(
                other_objective, best[other], method="L-BFGS-B", jac=True,
                bounds=[bounds[i] for i in np.flatnonzero(other)],
                options={"maxiter": 500, "ftol": 1e-12, "gtol": 1e-6},
            )
            if not profiled.success:
                profiled = minimize(
                    other_objective, profiled.x, method="SLSQP", jac=True,
                    bounds=[bounds[i] for i in np.flatnonzero(other)],
                    options={"maxiter": 500, "ftol": 1e-9},
                )
            if not profiled.success or not np.isfinite(profiled.fun):
                raise RuntimeError(f"SED bin {local_index} profile failed: {profiled.message}")
            return float(profiled.fun)

        ts = np.zeros(len(self.probe_k))
        lower68 = np.zeros(len(self.probe_k))
        upper68 = np.full(len(self.probe_k), np.nan)
        upper95 = np.full(len(self.probe_k), np.nan)
        for j, bin_index in enumerate(self.active_bins):
            null_nll = profile_nll(j, 0.0)
            ts[bin_index] = max(0.0, 2.0 * (null_nll - best_nll))
            if not intervals:
                continue

            def delta(value):
                return max(0.0, 2.0 * (profile_nll(j, value) - best_nll))

            if best[j] > 0 and ts[bin_index] > 1.0:
                lower_scaled = brentq(
                    lambda value: delta(value) - 1.0, 0.0, best[j],
                    xtol=1e-8, rtol=1e-7,
                )
                lower68[bin_index] = lower_scaled * self.k_scale[bin_index]

            def upper_root(target):
                lo = best[j]
                hi = min(self.upper_scaled[j], max(lo + 1.0, lo * 2.0))
                while delta(hi) < target and hi < self.upper_scaled[j]:
                    hi = min(self.upper_scaled[j], hi * 2.0)
                if delta(hi) < target:
                    return float("nan")
                return brentq(
                    lambda value: delta(value) - target, lo, hi,
                    xtol=1e-8, rtol=1e-7,
                ) * self.k_scale[bin_index]

            upper68[bin_index] = upper_root(1.0)
            upper95[bin_index] = upper_root(2.705543)
        result.ts = ts
        if intervals:
            result.lower68 = lower68
            result.upper68 = upper68
            result.upper95 = upper95
        return result


In [17]:
class WeakBinPoissonLike(PluginPrototype):
    """Precomputed detector-count likelihood for one weak SED-bin amplitude."""
    def __init__(self, name, base_counts, source_counts, observed, amplitude_parameter, total_source_counts):
        super().__init__(name, {})
        self.base = np.asarray(base_counts, dtype=float)
        self.template = np.asarray(source_counts, dtype=float)
        self.data = np.asarray(observed, dtype=float)
        self.amplitude_parameter = amplitude_parameter
        if np.any(self.base <= 0) or np.any(self.template < 0):
            raise ValueError("Invalid Bayesian Poisson template")
        self.reference = np.log(self.base + self.template)
        self.total_template = float(total_source_counts)

    def set_model(self, likelihood_model):
        self._likelihood_model = likelihood_model

    def get_log_like(self):
        x = float(self.amplitude_parameter.value)
        if x < 0:
            return -np.inf
        expected = self.base + x * self.template
        if np.any(expected <= 0) or np.any(~np.isfinite(expected)):
            return -np.inf
        # Constant terms cancel; nonresponsive occupied cells are omitted.
        return float(self.data @ (np.log(expected) - self.reference)
                     - x * self.total_template)

    def inner_fit(self):
        return self.get_log_like()

    def get_number_of_data_points(self):
        return int(self.data.size)


def bayesian_weak_limits(template_fit, fit_result, observed_hist, *, label):
    """Replace only TS<1 profile ULs with 95% threeML posterior quantiles."""
    observed = _dense_counts(observed_hist).ravel()
    active = template_fit.active_bins
    responsive = np.any(template_fit.count_templates > 0, axis=0)
    occupied = (observed > 0) & responsive
    count_templates = template_fit.count_templates[:, occupied]
    best = fit_result.scaled_k.copy()
    bayes_ul = np.full(len(template_fit.probe_k), np.nan)
    diagnostics = []
    for j, bin_index in enumerate(active):
        if fit_result.ts[bin_index] >= 1.0:
            continue
        source_all = count_templates[j]
        relevant = source_all > 0
        if not np.any(relevant):
            continue
        source = source_all[relevant]
        # Form the other-bin model directly. Subtracting one tiny component
        # from a large baseline can cancel the subnormal background to zero.
        other = np.arange(len(active)) != j
        base = (template_fit.background[occupied][relevant]
                + best[other] @ count_templates[other][:, relevant])
        if np.any(base <= 0):
            raise RuntimeError(f"{label}: nonpositive baseline in weak SED bin {bin_index+1}")
        profile_ul = fit_result.upper95[bin_index]
        scaled_profile = profile_ul / template_fit.k_scale[bin_index] if np.isfinite(profile_ul) else 0.
        prior_max = min(template_fit.upper_scaled[j], max(30., 20. * scaled_profile, 20. * best[j]))
        if not np.isfinite(prior_max) or prior_max <= 0:
            raise RuntimeError(f"{label}: invalid prior bound in SED bin {bin_index+1}")
        dummy = PointSource(f"weak_bin_{bin_index+1}", l=0., b=0.,
                            spectral_shape=Powerlaw(K=min(1., prior_max/2.),
                                                    piv=1000., index=-2.))
        posterior_model = Model(dummy)
        amplitude = posterior_model.point_sources[dummy.name].spectrum.main.Powerlaw.K
        for par in dummy.parameters.values():
            par.fix = True
        amplitude.fix = False
        # Powerlaw.K has a logarithmic transform in astromodels and therefore
        # needs a strictly positive numerical minimum. The uniform prior
        # still starts at zero; this 1e-12 floor has negligible mass.
        amplitude.min_value, amplitude.max_value = max(1e-12, 1e-12*prior_max), float(prior_max)
        amplitude.prior = Uniform_prior(lower_bound=0., upper_bound=float(prior_max))
        plugin = WeakBinPoissonLike(
            f"weak_bin_like_{bin_index+1}", base, source,
            observed[occupied][relevant], amplitude,
            total_source_counts=template_fit.count_totals[j],
        )
        analysis = BayesianAnalysis(posterior_model, DataList(plugin))
        analysis.set_sampler("emcee")
        analysis.sampler.setup(n_walkers=20, n_iterations=1800, n_burn_in=600,
                               seed=1400 + bin_index)
        analysis.sample(quiet=True)
        draws = np.asarray(analysis.samples[amplitude.path], dtype=float)
        q95 = float(np.quantile(draws, 0.95))
        if not np.isfinite(q95) or q95 >= 0.9 * prior_max:
            raise RuntimeError(f"{label}: weak-bin posterior hits its prior bound in bin {bin_index+1}")
        bayes_ul[bin_index] = q95 * template_fit.k_scale[bin_index]
        diagnostics.append(dict(bin=int(bin_index+1), TS=float(fit_result.ts[bin_index]),
                                prior_max_K=float(prior_max * template_fit.k_scale[bin_index]),
                                upper95_K=float(bayes_ul[bin_index]),
                                draws=int(draws.size)))
        print(f"{label} SED bin {bin_index+1}: TS={fit_result.ts[bin_index]:.2f}, "
              f"Bayesian 95% K UL={bayes_ul[bin_index]:.4g}")
    return bayes_ul, diagnostics


In [18]:
def _dense_counts(histogram):
    """Copy histogram values because response caches reuse their buffers."""
    values = histogram.to_dense(copy=False).contents
    if hasattr(values, "compute"):
        values = values.compute()
    if hasattr(values, "value"):
        values = values.value
    return np.asarray(values, dtype=float).copy()


def extract_binned_sed(model, global_shape, point_responses, background, *,
                       observed_hist, data_object, ordinary_psr, extended_psr, label, counts_scale=1.):
    """Fit independent NGC 4151 Ei-bin amplitudes for one catalog method."""
    edges = dr.axes["Ei"].edges.to_value(u.keV)
    if len(edges) != 11 or not np.allclose(edges[[0, -1]], [100., 10000.], rtol=1e-4):
        raise ValueError("Expected the native 10-bin, 0.1–10 MeV Ei response axis")
    pivots = np.sqrt(edges[:-1] * edges[1:])
    initial_k = np.asarray([global_shape.evaluate_at(e) for e in pivots], dtype=float)
    # Omit spectral_shape intentionally: BinnedSED defaults to a fixed E^-2
    # power law within every Ei bin. CPL values only initialize the optimizer.
    sed = BinnedSED.from_response(dr, initial_fluxes=initial_k)
    if not np.isclose(sed.spectral_shape.index.value, -2.0):
        raise RuntimeError("Expected BinnedSED default power-law index -2")
    source = model.ngc4151
    sed_source = PointSource(
        "ngc4151", l=source.position.l.value, b=source.position.b.value,
        spectral_shape=sed,
    )
    model.remove_source("ngc4151")
    model.add_source(sed_source)
    sed_shape = model.ngc4151.spectrum.main.shape
    for parameter in sed_shape.normalizations:
        parameter.value = 0.
        parameter.fix = True

    # A fresh folding object is essential: the global-fit folding object
    # caches its old ngc4151 source response under the same source name.
    sed_folding = BinnedThreeMLModelFolding(
        data=data_object, point_source_response=ordinary_psr, extended_source_response=extended_psr,
        source_specific_point_source_responses=point_responses,
    )
    sed_folding.set_model(model)
    fixed_sky = _dense_counts(sed_folding.expectation(copy=False))
    fixed_background = _dense_counts(background.expectation(copy=True))
    if not np.isfinite(counts_scale) or counts_scale <= 0:
        raise ValueError("counts_scale must be finite and positive")
    fixed_counts = counts_scale * (fixed_sky + fixed_background)
    # The source response is already bound to this model by expectation().
    # Read its counts directly. Subtracting two full-catalog predictions can
    # erase a faint NGC 4151 probe through floating-point cancellation.
    ngc_response = sed_folding._source_responses["ngc4151"]
    max_k = np.asarray([p.max_value for p in sed_shape.normalizations], dtype=float)
    probe_k = np.minimum(max_k, np.maximum(initial_k, 1e-5))
    templates = []
    for i, parameter in enumerate(sed_shape.normalizations):
        parameter.value = float(probe_k[i])
        template = _dense_counts(ngc_response.expectation(copy=False))
        if np.min(template) < -1e-7 * max(np.max(template), 1.):
            raise RuntimeError(f"Ei bin {i} produced negative source counts")
        templates.append(counts_scale * np.maximum(template, 0.))
        parameter.value = 0.
    template_fit = JointSEDTemplates(fixed_counts, np.asarray(templates), probe_k, max_k)
    fitted = template_fit.fit(
        _dense_counts(observed_hist), initial_k=initial_k, profile_ts=True, intervals=True,
    )
    bayes_upper95, bayes_diagnostics = bayesian_weak_limits(
        template_fit, fitted, observed_hist, label=label,
    )
    upper95 = np.asarray(fitted.upper95, dtype=float).copy()
    weak = fitted.ts < 1.0
    upper95[weak & np.isfinite(bayes_upper95)] = bayes_upper95[weak & np.isfinite(bayes_upper95)]
    result = {
        "bayesian upper 95": bayes_upper95,
        "bayesian diagnostics": bayes_diagnostics,
        "upper limit method": np.where(weak & np.isfinite(bayes_upper95),
                                       "Bayesian 95%", "profile 95%"),
        "edges keV": edges, "pivot keV": pivots, "K": fitted.k,
        "TS": fitted.ts, "lower 68": fitted.lower68,
        "upper 68": fitted.upper68, "upper 95": upper95,
        "spectral shape": "Powerlaw(index=-2)",
    }
    print(f"SED: {np.sum(fitted.ts >= 4)} bins at TS >= 4 of {len(pivots)} Ei bins")
    return result


## Full-mock parallel-chunks catalog CPL fit

This is the only full-mock source-catalog fit. Its NGC 4151 component is a free CPL; all other catalog sources retain their injected parameters.

The fit cell prints each free parameter with a covariance-based 68% interval. If threeML’s bound-truncated covariance samples do not enclose the maximum-likelihood value, the code reports a local one-sigma covariance interval in the parameter’s transformed coordinates. These are approximate intervals, particularly when the cutoff approaches a fit bound.

The free CPL exponent is restricted to −5 ≤ `index` ≤ −0.01, so positive photon indices are allowed and the injected `index = −1.75` remains in range.

**Band convention:** as in `NGC4151_DC4_Parallel_vs_FitNorm_SED_Plots.ipynb`, the blue band propagates samples from the fitted joint covariance through the CPL and takes a pointwise 68% equal-tail interval. This is an approximate covariance interval, not a profile-likelihood interval; it may be unstable near a parameter bound. It is independent of the SED-point intervals. Changing the index bound requires rerunning the catalog fit, SED, and final plot.

In the saved refit, the catalog CPL index reached −0.01 and threeML rejected about 53% of covariance draws at parameter bounds. The displayed band follows the requested equal-tail method, but it should not be interpreted as a reliable 68% physical constraint on that boundary fit.


In [ ]:
def fit_parallel_catalog(*, observed_data=None, point_responses_override=None):
    fit_data = data if observed_data is None else observed_data
    if fit_data.axes != data.axes:
        raise ValueError("The updated mock has different detector axes")
    method = "parallel chunks"
    catalog_path = parallel_catalog_path
    catalog_sha = hashlib.sha256(catalog_path.read_bytes()).hexdigest()
    model = make_fit_model(deepcopy(parallel_model))
    background = FreeNormBinnedBackground(
        {"total_bkg": total_bkg}, sc_history=sc_orientation, copy=True,
    )
    if point_responses_override is None:
        point_responses, temporal_diagnostics = make_temporal_responses(
            model, parallel_recipe, sc_orientation_cut, source_gti, fit_data,
            instrument_response, dr, exposure_multiplier=exposure_multiplier,
            full_history=sc_orientation_full,
        )
    else:
        point_responses = dict(point_responses_override)
        if any(response.axes != fit_data.axes for response in point_responses.values()):
            raise ValueError("Cached temporal responses have different detector axes")
        temporal_diagnostics = []
    folding = BinnedThreeMLModelFolding(
        data=fit_data, point_source_response=psr, extended_source_response=esr,
        source_specific_point_source_responses=point_responses,
    )
    likelihood = PoissonLikelihood(fit_data, folding, background)
    plugin = ThreeMLPluginInterface("cosi", likelihood, folding, background)
    plugin.bkg_parameter["total_bkg"] = Parameter(
        "total_bkg", background_rate_initial, min_value=0., max_value=50.,
        delta=max(0.05 * background_rate_initial, 1e-6), unit=u.Hz,
    )
    joint = JointLikelihood(model, DataList(plugin), verbose=False)
    initial_loglike = float(plugin.get_log_like())
    covariance_valid = fit_joint_with_fallback(joint, method)
    if float(plugin.get_log_like()) < initial_loglike - 1e-3:
        raise RuntimeError("Catalog optimization worsened the Poisson likelihood")
    results = joint.results
    shape = results.optimized_model.ngc4151.spectrum.main.Cutoff_powerlaw
    flux = np.asarray([shape.evaluate_at(e) for e in energy], dtype=float)
    low = high = None
    index_at_bound = abs(float(shape.index.value) - float(shape.index.max_value)) < 1e-2
    if covariance_valid:
        # Match NGC4151_DC4_Parallel_vs_FitNorm_SED_Plots.ipynb:
        # sample the fitted joint covariance (including parameter correlations),
        # propagate the CPL, and take a pointwise 68% equal-tail interval.
        try:
            def cpl_flux(e, K, index, xc):
                return K * (e / shape.piv.value)**index * np.exp(-e / xc)
            propagated = results.propagate(
                cpl_flux,
                K=results.get_variates(shape.K.path),
                index=results.get_variates(shape.index.path),
                xc=results.get_variates(shape.xc.path),
            )
            interval = np.array([
                propagated(e).equal_tail_interval(cl=0.68) for e in energy
            ])
            low, high = interval[:, 0], interval[:, 1]
        except (BadCovariance, CannotComputeCovariance) as exc:
            print(f"{method}: 68% global band unavailable ({type(exc).__name__})")
    output = {
        "fit flux": flux, "fit energy keV": energy.copy(),
        "fit flux low 68": low, "fit flux high 68": high,
        "K": float(shape.K.value), "index": float(shape.index.value),
        "xc (keV)": float(shape.xc.value),
        "background (Hz)": float(plugin.bkg_parameter["total_bkg"].value),
        "log likelihood": float(plugin.get_log_like()),
        "CPL index lower bound": -5.,
        "CPL index upper bound": -0.01,
        "CPL index at bound": index_at_bound,
        "parameter intervals 68": parameter_intervals_68(
            results, {"K": shape.K, "index": shape.index, "xc (keV)": shape.xc,
                      "background (Hz)": plugin.bkg_parameter["total_bkg"]},
            covariance_valid,
        ),
        "catalog file": str(catalog_path.resolve()), "catalog sha256": catalog_sha,
        "temporal diagnostics": pd.DataFrame(temporal_diagnostics),
        "model for sed": model, "shape for sed": shape,
        "point responses for sed": point_responses,
        "background for sed": background,
    }
    return output


In [21]:
refresh_dc4_response_integrators()
parallel_result = fit_parallel_catalog()
print(f"Parallel catalog CPL log L = {parallel_result['log likelihood']:.6f}")
print("Fitted parameters (covariance-based 68% intervals):")
for name, spec in (("K", ".4g"), ("index", ".3g"),
                   ("xc (keV)", ".4g"), ("background (Hz)", ".5g")):
    print(f"  {name}: {format_parameter_68(parallel_result, name, spec)}")
if parallel_result["index"] > 0 or parallel_result["xc (keV)"] < 1.1 * 100.:
    print("Diagnostic: this CPL is hard and its cutoff is close to the lower "
          "bound. Check alternative starts and residual counts before "
          "interpreting its shape physically.")


index: parameter is at a bound; two-sided 68% interval unavailable
Parallel catalog CPL log L = 118103360.937088
Fitted parameters (covariance-based 68% intervals):
  K: 1.093e-05 (−3.72e-06/+5.592e-06)
  index: -0.01 (68% unavailable)
  xc (keV): 141.8 (−24.42/+28.53)
  background (Hz): 25.523 (−0.0068672/+0.0069792)


## Catalog SED and comparison fits

The full-mock and isolated comparison CPL fits both use three-month histograms. Neither comparison CPL fit has independent SED points. The comparison curves use the same three-month exposure as the catalog fit.


In [23]:
if (parallel_result.get("CPL index lower bound") != -5. or parallel_result.get("CPL index upper bound") != -0.01):
    raise RuntimeError("Rerun the catalog CPL fit cell with index < 0 before rebuilding its SED")
parallel_result["sed"] = extract_binned_sed(
    parallel_result["model for sed"], parallel_result["shape for sed"],
    parallel_result["point responses for sed"], parallel_result["background for sed"],
    observed_hist=data_hist, data_object=data, ordinary_psr=psr,
    extended_psr=esr, label="full-mock catalog",
)


full-mock catalog SED bin 1: TS=0.12, Bayesian 95% K UL=4.739e-06
full-mock catalog SED bin 8: TS=0.29, Bayesian 95% K UL=5.47e-08
full-mock catalog SED bin 9: TS=0.00, Bayesian 95% K UL=2.87e-08
full-mock catalog SED bin 10: TS=0.00, Bayesian 95% K UL=8.98e-09
SED: 3 bins at TS >= 4 of 10 Ei bins


In [24]:
# The red comparison curve in Figure 1 uses the same three-month exposure
# as the SC and No SC curves. Its observed histogram is source + background only.
comparison_data_hist = deepcopy(comparison_data_hist_three_month)
comparison_bkg_hist = total_bkg
comparison_sc_orientation = sc_orientation_cut
comparison_background_rate_initial = background_rate_initial
comparison_data = EmCDSBinnedData(comparison_data_hist)
comparison_instrument_response = BinnedInstrumentResponse(dr, comparison_data)
comparison_psr = BinnedThreeMLPointSourceResponse(
    data=comparison_data, instrument_response=comparison_instrument_response,
    sc_history=comparison_sc_orientation, energy_axis=dr.axes["Ei"],
    polarization_axis=dr.axes["Pol"] if "Pol" in dr.axes.labels else None,
    nside=2 * comparison_data.axes["PsiChi"].nside,
)
print("Three-month isolated comparison counts:",
      float(comparison_data_hist.to_dense(copy=False).contents.sum()))

def fit_one_source_cpl(*, observed_data=None, source_file=None,
                            cache_name="NGC4151_DC4_paper_isolated_cpl_original_fits_3months.json"):
    """Fit a selected three-month source + background histogram."""
    fit_data = comparison_data if observed_data is None else observed_data
    source_file = ngc_original_fits_path if source_file is None else Path(source_file)
    if fit_data.axes != comparison_data.axes:
        raise ValueError("The comparison histogram has different detector axes")
    source = parallel_model.ngc4151
    injected_thermal = source.spectrum.main.composite
    cpl = Cutoff_powerlaw(
        K=injected_thermal.K_1.value, piv=injected_thermal.piv_1.value,
        index=injected_thermal.index_1.value, xc=injected_thermal.xc_1.value,
    )
    point = PointSource(
        "ngc4151", l=source.position.l.value, b=source.position.b.value,
        spectral_shape=cpl,
    )
    one_model = Model(point)
    shape = one_model.ngc4151.spectrum.main.Cutoff_powerlaw
    for parameter in point.parameters.values():
        parameter.fix = True
    shape.K.min_value, shape.K.max_value = 1e-8, 1e-2
    shape.K.delta, shape.K.fix = 0.1 * shape.K.value, False
    # astromodels uses a signed exponent: photon index Γ>0 means index<0.
    # Keep the injected index -1.75 inside the allowed scan.
    shape.index.min_value, shape.index.max_value = -5., -0.01
    shape.index.delta, shape.index.fix = 0.1, False
    shape.xc.min_value, shape.xc.max_value = 100., 10000.
    shape.xc.delta, shape.xc.fix = 0.1 * shape.xc.value, False
    shape.piv.fix = True
    if len(one_model.free_parameters) != 3:
        raise RuntimeError("NGC-only fit must free only CPL K, index, and xc")
    background = FreeNormBinnedBackground(
        {"total_bkg": comparison_bkg_hist}, sc_history=comparison_sc_orientation, copy=True,
    )
    folding = BinnedThreeMLModelFolding(
        data=fit_data, point_source_response=comparison_psr,
    )
    likelihood = PoissonLikelihood(fit_data, folding, background)
    plugin = ThreeMLPluginInterface("cosi_one", likelihood, folding, background)
    plugin.bkg_parameter["total_bkg"] = Parameter(
        "total_bkg", comparison_background_rate_initial, min_value=0., max_value=50.,
        delta=max(0.05 * comparison_background_rate_initial, 1e-6), unit=u.Hz,
    )
    joint = JointLikelihood(one_model, DataList(plugin), verbose=False)
    covariance_valid = fit_joint_with_fallback(joint, "NGC-only + background CPL")
    log_likelihood = float(plugin.get_log_like())
    fitted = joint.results.optimized_model.ngc4151.spectrum.main.Cutoff_powerlaw
    best_flux = np.asarray([fitted.evaluate_at(e) for e in energy], dtype=float)
    low = high = None
    if covariance_valid:
        try:
            def cpl_flux(e, K, index, xc):
                return K * (e / fitted.piv.value)**index * np.exp(-e / xc)
            propagated = joint.results.propagate(
                cpl_flux,
                K=joint.results.get_variates(fitted.K.path),
                index=joint.results.get_variates(fitted.index.path),
                xc=joint.results.get_variates(fitted.xc.path),
            )
            interval = np.array([
                propagated(e).equal_tail_interval(cl=0.68) for e in energy
            ])
            low, high = interval[:, 0], interval[:, 1]
        except (BadCovariance, CannotComputeCovariance) as exc:
            print(f"NGC-only 68% global band unavailable ({type(exc).__name__})")
    result = {
        "fit flux": best_flux, "fit energy keV": energy.copy(),
        "fit flux low 68": low, "fit flux high 68": high,
        "K": float(fitted.K.value), "index": float(fitted.index.value),
        "xc (keV)": float(fitted.xc.value),
        "background (Hz)": float(plugin.bkg_parameter["total_bkg"].value),
        "log likelihood": log_likelihood,
        "covariance valid": bool(covariance_valid),
        "parameter intervals 68": parameter_intervals_68(
            joint.results,
            {"K": fitted.K, "index": fitted.index, "xc (keV)": fitted.xc,
             "background (Hz)": plugin.bkg_parameter["total_bkg"]},
            covariance_valid,
        ),
        "source data file": str(source_file.resolve()),
        "background data file": str(background_path.resolve()),
        "exposure months": 3,
    }
    print("NGC-only + background CPL fit:", {
        k: result[k] for k in ("K", "index", "xc (keV)", "background (Hz)")
    })
    parameter_file = notebook_dir / "plots" / cache_name
    parameter_file.parent.mkdir(parents=True, exist_ok=True)
    parameter_file.write_text(json.dumps({
        k: result[k] for k in (
            "K", "index", "xc (keV)", "background (Hz)", "log likelihood",
            "covariance valid", "parameter intervals 68",
            "source data file", "background data file", "exposure months",
        )
    }, indent=2) + "\n")
    print("Saved fitted parameters:", parameter_file)
    return result


one_source_fit = fit_one_source_cpl()
gc.collect()


Three-month isolated comparison counts: 25125949.0
NGC-only + background CPL fit: {'K': 9.085495157672528e-06, 'index': -2.022140675149647, 'xc (keV)': 331.40519840723493, 'background (Hz)': 25.568459742882272}
Saved fitted parameters: /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/plots/NGC4151_DC4_paper_isolated_cpl_original_fits_3months.json


In [25]:
# Fit an NGC 4151-only model to the full mock, retaining every injected source
# in the DATA while omitting the other sources from this comparison MODEL.
def fit_ngc_on_full_mock_cpl(*, observed_data=None, source_file=None,
                                  cache_name="NGC4151_DC4_paper_full_mock_no_catalog_comparison.json"):
    fit_data = data if observed_data is None else observed_data
    source_file = mock_data_path if source_file is None else Path(source_file)
    if fit_data.axes != data.axes:
        raise ValueError("The full-mock comparison histogram has different detector axes")
    source = parallel_model.ngc4151
    injected_thermal = source.spectrum.main.composite
    cpl = Cutoff_powerlaw(
        K=injected_thermal.K_1.value, piv=injected_thermal.piv_1.value,
        index=injected_thermal.index_1.value, xc=injected_thermal.xc_1.value,
    )
    point = PointSource(
        "ngc4151", l=source.position.l.value, b=source.position.b.value,
        spectral_shape=cpl,
    )
    model = Model(point)
    shape = model.ngc4151.spectrum.main.Cutoff_powerlaw
    for parameter in point.parameters.values():
        parameter.fix = True
    shape.K.min_value, shape.K.max_value = 1e-8, 1e-2
    shape.K.delta, shape.K.fix = 0.1 * shape.K.value, False
    # astromodels uses a signed exponent: photon index Γ>0 means index<0.
    # Keep the injected index -1.75 inside the allowed scan.
    shape.index.min_value, shape.index.max_value = -5., -0.01
    shape.index.delta, shape.index.fix = 0.1, False
    shape.xc.min_value, shape.xc.max_value = 100., 10000.
    shape.xc.delta, shape.xc.fix = 0.1 * shape.xc.value, False
    shape.piv.fix = True
    if len(model.free_parameters) != 3:
        raise RuntimeError("Full-mock NGC-only fit must free only CPL K, index, and xc")

    # data and psr are the ordinary full-mock objects shared by catalog fits.
    background = FreeNormBinnedBackground(
        {"total_bkg": total_bkg}, sc_history=sc_orientation, copy=True,
    )
    folding = BinnedThreeMLModelFolding(data=fit_data, point_source_response=psr)
    likelihood = PoissonLikelihood(fit_data, folding, background)
    plugin = ThreeMLPluginInterface("cosi_full_mock_ngc", likelihood, folding, background)
    plugin.bkg_parameter["total_bkg"] = Parameter(
        "total_bkg", background_rate_initial, min_value=0., max_value=50.,
        delta=max(0.05 * background_rate_initial, 1e-6), unit=u.Hz,
    )
    joint = JointLikelihood(model, DataList(plugin), verbose=False)
    covariance_valid = fit_joint_with_fallback(joint, "NGC-only model on full mock")
    log_likelihood = float(plugin.get_log_like())
    fitted = joint.results.optimized_model.ngc4151.spectrum.main.Cutoff_powerlaw
    best_flux = np.asarray([fitted.evaluate_at(e) for e in energy], dtype=float)
    low = high = None
    if covariance_valid:
        try:
            def cpl_flux(e, K, index, xc):
                return K * (e / fitted.piv.value)**index * np.exp(-e / xc)
            propagated = joint.results.propagate(
                cpl_flux,
                K=joint.results.get_variates(fitted.K.path),
                index=joint.results.get_variates(fitted.index.path),
                xc=joint.results.get_variates(fitted.xc.path),
            )
            interval = np.array([
                propagated(e).equal_tail_interval(cl=0.68) for e in energy
            ])
            low, high = interval[:, 0], interval[:, 1]
        except (BadCovariance, CannotComputeCovariance) as exc:
            print(f"Full-mock NGC-only 68% band unavailable ({type(exc).__name__})")
    result = {
        "fit flux": best_flux, "fit energy keV": energy.copy(),
        "fit flux low 68": low, "fit flux high 68": high,
        "K": float(fitted.K.value), "index": float(fitted.index.value),
        "xc (keV)": float(fitted.xc.value),
        "background (Hz)": float(plugin.bkg_parameter["total_bkg"].value),
        "log likelihood": log_likelihood,
        "covariance valid": bool(covariance_valid),
        "parameter intervals 68": parameter_intervals_68(
            joint.results,
            {"K": fitted.K, "index": fitted.index, "xc (keV)": fitted.xc,
             "background (Hz)": plugin.bkg_parameter["total_bkg"]},
            covariance_valid,
        ),
        "source data file": str(source_file.resolve()),
        "background data file": str(background_path.resolve()),
    }
    print("NGC-only model on full mock CPL fit:", {
        k: result[k] for k in ("K", "index", "xc (keV)", "background (Hz)")
    })
    parameter_file = notebook_dir / "plots" / cache_name
    parameter_file.parent.mkdir(parents=True, exist_ok=True)
    parameter_file.write_text(json.dumps({
        k: result[k] for k in (
            "K", "index", "xc (keV)", "background (Hz)", "log likelihood",
            "covariance valid", "parameter intervals 68",
            "source data file", "background data file",
        )
    }, indent=2) + "\n")
    print("Saved fitted parameters:", parameter_file)
    return result


full_mock_ngc_fit = fit_ngc_on_full_mock_cpl()
gc.collect()


xc (keV): parameter is at a bound; two-sided 68% interval unavailable
NGC-only model on full mock CPL fit: {'K': 1.1032576663394498e-05, 'index': -2.2180104794752475, 'xc (keV)': 9999.997124970403, 'background (Hz)': 26.521349487950097}
Saved fitted parameters: /Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/plots/NGC4151_DC4_paper_full_mock_no_catalog_comparison.json


## Figure 1: parallel-chunks source catalog on the original full mock

The red isolated comparison uses the original NGC 4151 event FITS plus background, while the older comparison notebook used the modified 0.30 source file. It is a CPL-only fit. The black dotted curve is the original **total** tabulated spectrum, so the fitted CPL is also compared with a separate dashed thermal-CPL reference. The original source contributes 54,915 GTI-selected counts, versus 76,103 in the modified 0.30 source histogram. The difference therefore affects the low-energy fit too; it is not just a visually small high-energy tail. A remaining low-energy discrepancy needs a residual check.
If the free CPL index reaches its upper bound (−1), the fit reports no two-sided covariance interval or 68% spectral band for the SC curve. A boundary fit requires a profile-likelihood check before quoting an index uncertainty. The optional three-source normalization test in `NGC4151_DC4_OriginalMock_CrabNorm_NGC_CPL_Multistart.ipynb` improves the likelihood but does not move the index away from this bound.


In [31]:
from agn_paper_plot_style import BLUE, GREEN, ink, paper_style, style_axis

PAPER_OUTPUT_DIR = Path.home() / (
    "Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/DC4_Paper"
)
SAVE_PLOTS = True
FONT_SIZE = 25
PLOT_ENERGY_KEV = np.geomspace(100., 10000., 400)
KEV_TO_ERG = u.keV.to(u.erg)
FULL_MOCK_COLOR = GREEN  # NGC-only model on full mock data.
ISOLATED_COLOR = "#E64949"  # NGC 4151 plus background only.
METHOD_LABELS = {"parallel chunks": "SC"}

def fitted_cpl_flux(fit, energies_keV, pivot_keV):
    """Evaluate a saved CPL fit on a fresh plot grid without refitting."""
    shape = Cutoff_powerlaw(
        K=fit["K"], piv=pivot_keV, index=fit["index"], xc=fit["xc (keV)"],
    )
    return np.asarray([shape.evaluate_at(e) for e in energies_keV], dtype=float)


def plot_method_sed(method, filename_stem, method_color, *, context=None):
    if context is None:
        context = {
            "catalog fit": fit_outputs[method],
            "mock path": mock_data_path,
            "full mock comparison": full_mock_ngc_fit,
            "isolated comparison": one_source_fit,
            "isolated source path": ngc_original_fits_path,
            "injected flux": original_injected_flux,
            "isolated label": "Isolated NGC 4151 (3 mo) best fit & 68% band",
            "annotation": "NGC 4151\n3 months",
        }
    output = context["catalog fit"]
    plot_mock_path = Path(context["mock path"])
    plot_full_mock_fit = context["full mock comparison"]
    plot_isolated_fit = context["isolated comparison"]
    plot_isolated_path = Path(context["isolated source path"])
    plot_injected_flux = context["injected flux"]
    if method == "parallel chunks" and (output.get("CPL index lower bound") != -5. or output.get("CPL index upper bound") != -0.01):
        raise RuntimeError("Rerun the catalog CPL fit and SED with index < 0 before plotting")
    expected_catalog = parallel_catalog_path
    expected_sha = hashlib.sha256(expected_catalog.read_bytes()).hexdigest()
    if (output.get("catalog file") != str(expected_catalog.resolve())
            or output.get("catalog sha256") != expected_sha):
        raise RuntimeError(
            f"{method}: the saved fit predates the current {expected_catalog.name}; "
            "rerun this method's fit cell before plotting"
        )
    if (plot_full_mock_fit.get("source data file") != str(plot_mock_path.resolve())
            or plot_full_mock_fit.get("background data file") != str(background_path.resolve())):
        raise RuntimeError("Rerun the NGC-only fit cell on the full mock data")
    if (plot_isolated_fit.get("source data file") != str(plot_isolated_path.resolve())
            or plot_isolated_fit.get("background data file") != str(background_path.resolve())
            or plot_isolated_fit.get("exposure months") != 3):
        raise RuntimeError("Rerun the three-month NGC-only fit on the original event FITS")
    sed = output["sed"]
    if sed.get("spectral shape") != "Powerlaw(index=-2)":
        raise RuntimeError(f"{method}: rerun the fit cell to use the default E^-2 SED shape")
    pivot = np.asarray(sed["pivot keV"], dtype=float)
    edges = np.asarray(sed["edges keV"], dtype=float)
    if len(pivot) != 10 or len(edges) != 11:
        raise ValueError("This figure expects all 10 native Ei response bins")
    detected = (sed["TS"] >= 4.) & (sed["K"] > 0.) & (
        sed["lower 68"] > 0.) & np.isfinite(sed["upper 68"])
    upper = (~detected) & np.isfinite(sed["upper 95"]) & (sed["upper 95"] > 0.)
    plot_energy = PLOT_ENERGY_KEV
    x = plot_energy / 1000.
    method_pivot = method_models[method].ngc4151.spectrum.main.composite.piv_1.value
    comparison_pivot = parallel_model.ngc4151.spectrum.main.composite.piv_1.value
    method_name = METHOD_LABELS[method]
    # Display the pointwise equal-tail band when propagation succeeded.
    show_catalog_band = output["fit flux low 68"] is not None

    with paper_style("light", FONT_SIZE):
        fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
        style_axis(ax, "light", FONT_SIZE)
        injected = plot_injected_flux(plot_energy)
        ax.plot(x, KEV_TO_ERG * plot_energy**2 * injected, color=ink("light"), ls=":", lw=3,
                label="Injected total")
        ax.plot(x, KEV_TO_ERG * plot_energy**2 * fitted_cpl_flux(output, plot_energy, method_pivot),
                color=method_color, lw=2.5,
                label=(f"{method_name} best fit & 68% band"
                       if show_catalog_band
                       else f"{method_name} best fit"))
        if show_catalog_band:
            # The saved band may have been calculated on the former 0.2–5 MeV
            # grid. Preserve its true energy coordinates instead of stretching
            # uncertainty estimates into bins where they were not calculated.
            fit_energy = np.asarray(output.get(
                "fit energy keV",
                np.geomspace(200., 5000., len(output["fit flux"])),
            ), dtype=float)
            low = np.where(output["fit flux low 68"] > 0,
                           KEV_TO_ERG * fit_energy**2 * output["fit flux low 68"], np.nan)
            high = np.where(output["fit flux high 68"] > 0,
                            KEV_TO_ERG * fit_energy**2 * output["fit flux high 68"], np.nan)
            ax.fill_between(fit_energy / 1000., low, high, color=method_color,
                            alpha=0.27, linewidth=0)
        # Two one-source CPL models use different DATA: the complete mock and
        # the isolated NGC 4151 plus instrumental background histogram.
        ax.plot(x, KEV_TO_ERG * plot_energy**2 * fitted_cpl_flux(
            plot_full_mock_fit, plot_energy, comparison_pivot),
            color=FULL_MOCK_COLOR, ls="--", lw=2.5,
            label="No SC best-fit & 68% band")
        if plot_full_mock_fit["fit flux low 68"] is not None:
            full_e = np.asarray(plot_full_mock_fit["fit energy keV"], dtype=float)
            full_low = KEV_TO_ERG * full_e**2 * np.where(
                plot_full_mock_fit["fit flux low 68"] > 0,
                plot_full_mock_fit["fit flux low 68"], np.nan,
            )
            full_high = KEV_TO_ERG * full_e**2 * np.where(
                plot_full_mock_fit["fit flux high 68"] > 0,
                plot_full_mock_fit["fit flux high 68"], np.nan,
            )
            ax.fill_between(full_e / 1000., full_low, full_high,
                            color=FULL_MOCK_COLOR, alpha=0.15, linewidth=0)
        ax.plot(x, KEV_TO_ERG * plot_energy**2 * fitted_cpl_flux(
            plot_isolated_fit, plot_energy, comparison_pivot), color=ISOLATED_COLOR,
            ls="-.", lw=2.5, label=context["isolated label"])
        if plot_isolated_fit["fit flux low 68"] is not None:
            one_e = np.asarray(plot_isolated_fit["fit energy keV"], dtype=float)
            one_low = KEV_TO_ERG * one_e**2 * np.where(
                plot_isolated_fit["fit flux low 68"] > 0,
                plot_isolated_fit["fit flux low 68"], np.nan,
            )
            one_high = KEV_TO_ERG * one_e**2 * np.where(
                plot_isolated_fit["fit flux high 68"] > 0,
                plot_isolated_fit["fit flux high 68"], np.nan,
            )
            ax.fill_between(one_e / 1000., one_low, one_high, color=ISOLATED_COLOR,
                            alpha=0.17, linewidth=0)

        if np.any(detected):
            k = sed["K"][detected]
            p = pivot[detected]
            ax.errorbar(p / 1000., KEV_TO_ERG * p**2 * k,
                        xerr=np.vstack((p - edges[:-1][detected],
                                        edges[1:][detected] - p)) / 1000.,
                        yerr=np.vstack((KEV_TO_ERG * p**2 * (k - sed["lower 68"][detected]),
                                        KEV_TO_ERG * p**2 * (sed["upper 68"][detected] - k))),
                        fmt="o", color=method_color, ecolor=method_color,
                        markerfacecolor="white", markeredgewidth=1.8,
                        markersize=9, capsize=4, elinewidth=1.5,
                        label="COSI SED")
        if np.any(upper):
            p = pivot[upper]
            limit = KEV_TO_ERG * p**2 * sed["upper 95"][upper]
            ax.errorbar(p / 1000., limit,
                        xerr=np.vstack((p - edges[:-1][upper],
                                        edges[1:][upper] - p)) / 1000.,
                        yerr=np.maximum(0.5 * limit, np.finfo(float).tiny),
                        uplims=True, fmt="v", color=method_color,
                        markerfacecolor="white", markeredgewidth=1.8,
                        markersize=10, capsize=5, elinewidth=1.8)

        visible_limits = KEV_TO_ERG * pivot[upper]**2 * sed["upper 95"][upper]
        visible_points = KEV_TO_ERG * pivot[detected]**2 * sed["K"][detected]
        visible = np.r_[visible_limits, visible_points]
        y_min = min(1e-4 * KEV_TO_ERG, 0.35 * np.min(visible)) if visible.size else 1e-2 * KEV_TO_ERG
        y_max = max(KEV_TO_ERG*3, 2 * np.max(visible)) if visible.size else KEV_TO_ERG
        ax.set(xscale="log", yscale="log", xlim=(0.1, 10.), ylim=(y_min, y_max),
               xlabel="Energy (MeV)",
               ylabel=r"Energy Flux (erg cm$^{-2}$ s$^{-1}$)")
        ax.xaxis.set_major_locator(mticker.FixedLocator([0.1, 1.0, 10.0]))
        ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.1", "1", "10"]))
        ax.text(0.97, 0.97, context["annotation"],
                transform=ax.transAxes, ha="right", va="top", fontsize=FONT_SIZE,
                color=ink("light"),
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=5))
        ax.legend(loc="lower left", frameon=False, fontsize=FONT_SIZE)
        if SAVE_PLOTS:
            PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            pdf_path = PAPER_OUTPUT_DIR / f"{filename_stem}_white.pdf"
            png_path = PAPER_OUTPUT_DIR / f"{filename_stem}_white.png"
            fig.savefig(pdf_path, format="pdf", facecolor="white",
                        edgecolor="white", transparent=False, bbox_inches="tight")
            fig.savefig(png_path, dpi=220, facecolor="white",
                        edgecolor="white", transparent=False, bbox_inches="tight")
            print("Saved", pdf_path, "and", png_path)
        plt.show()
    return fig


In [32]:
fit_outputs = {"parallel chunks": parallel_result}
parallel_figure = plot_method_sed(
    "parallel chunks", "NGC4151_DC4_paper_parallel_catalog", BLUE,
)


RuntimeError: Rerun the catalog CPL fit and SED with index < 0 before plotting

## 24-month simulated full-catalog forecast

The `Paper_Plots_DC4` branch has two distinct 24-month constructions. The thermal paper notebook scales its **unfluctuated** isolated-source and background histograms by eight; its `Fluctuate_True` notebooks draw new Poisson counts from scaled expectations. Both use isolated NGC 4151 data, so their fit values are not directly comparable to this 71-entry source-catalog fit. Scaling one unchanged data realization and its prediction cannot, by itself, move an exact maximum-likelihood best fit; a reported change can also reflect optimizer convergence or other model/data differences.

No independent 24-month **full DC4 mock** is supplied here. This cell therefore creates a clearly labeled forecast: it folds the original injected NGC 4151 `.dat` spectrum and the other fixed catalog sources through the existing NGC 4151 GTI responses, multiplies the source and background expected counts by eight, and uses the scaled, unfluctuated expectation as the default Asimov data set. Set `FLUCTUATE_24M = True` only to draw a new Poisson realization, as in the `Fluctuate_True` notebooks. The 24-month likelihood multiplies model expectations by the same factor. The original three-month data are **not** multiplied or reused as 24-month observations. This forecast assumes that the selected three-month pointing, GTI, light curves, and background rate repeat eight times.

Because the new data are generated from the fixed catalog model, this forecast tests statistical exposure scaling under that model; it cannot establish that a real 24-month DC4 mock would remove mismatches present in the original three-month data.

The CPL fit still omits NGC 4151's injected power-law tail. The output includes SED points from the chosen forecast data and a white-background paper plot. These are forecast points, not measured DC4 counts. In the optional Poisson mode, a different seed can shift the best fit; it is not guaranteed to be closer to the injected shape for every realization.

In [ ]:
# Construct a fresh 24-month realization from full-catalog expectations.
# Match the thermal Paper_Plots_DC4 construction by scaling source and
# background expectations. Optional fluctuations follow Fluctuate_True.
# This remains a forecast: DC4 provides no 24-month full mock here.
EXPOSURE_SCALE_24M = 8
SIMULATION_SEED_24M = 4151
FLUCTUATE_24M = False  # True draws one new Poisson realization.
if exposure_multiplier != 1:
    raise ValueError("Run the three-month base notebook first (exposure_multiplier=1)")
if parallel_result.get("CPL index upper bound") != -0.01:
    raise RuntimeError("Run the current three-month catalog fit before the forecast")

# SpecFromDat internally divides the tabulated spectrum by a left-rectangle
# sum. Supply that same sum as K to reproduce the original .dat amplitudes.
dat_bin_widths = np.diff(original_energy_keV, append=original_energy_keV[-1])
dat_normalization = float(np.sum(original_photon_flux * dat_bin_widths))
if not np.isfinite(dat_normalization) or dat_normalization <= 0:
    raise ValueError("The original NGC 4151 .dat spectrum is invalid")
injected_shape_24m = SpecFromDat(
    K=dat_normalization, dat=str(original_ngc_spectrum_path),
)
assert np.allclose(
    np.asarray([injected_shape_24m.evaluate_at(e) for e in original_energy_keV]),
    original_photon_flux, rtol=1e-9, atol=1e-20,
), "SpecFromDat must reproduce the injected .dat samples"

injected_model_24m = deepcopy(parallel_model)
ngc_catalog_source = injected_model_24m.ngc4151
injected_source_24m = PointSource(
    "ngc4151", l=ngc_catalog_source.position.l.value,
    b=ngc_catalog_source.position.b.value,
    spectral_shape=injected_shape_24m,
)
for parameter in injected_source_24m.parameters.values():
    parameter.fix = True
injected_model_24m.remove_source("ngc4151")
injected_model_24m.add_source(injected_source_24m)
if len(injected_model_24m.sources) != 71 or injected_model_24m.free_parameters:
    raise RuntimeError("Expected 71 frozen injected catalog sources")

# The response is a *three-month* GTI response. Reuse its source-specific
# light-curve weights, then scale the complete detector expectation by eight.
injection_folding_24m = BinnedThreeMLModelFolding(
    data=data, point_source_response=psr, extended_source_response=esr,
    source_specific_point_source_responses=parallel_result["point responses for sed"],
)
injection_folding_24m.set_model(injected_model_24m)
source_expectation_3m = _dense_counts(injection_folding_24m.expectation(copy=False))
background_expectation_3m = _dense_counts(total_bkg)
mean_counts_24m = EXPOSURE_SCALE_24M * (
    source_expectation_3m + background_expectation_3m
)
if (np.any(~np.isfinite(mean_counts_24m))
        or np.any(mean_counts_24m < -1e-8)):
    raise ValueError("The 24-month Poisson expectations are invalid")
mean_counts_24m = np.maximum(mean_counts_24m, 0.)
rng_24m = np.random.default_rng(SIMULATION_SEED_24M)
simulated_hist_24m = Histogram(data_hist.axes)
simulated_hist_24m[:] = (
    rng_24m.poisson(mean_counts_24m) if FLUCTUATE_24M else mean_counts_24m
)
simulated_data_24m = EmCDSBinnedData(simulated_hist_24m)
print("24-month forecast:", "Poisson" if FLUCTUATE_24M else "Asimov",
      "seed:", SIMULATION_SEED_24M if FLUCTUATE_24M else "unused")
print("Expected / realized counts:", int(round(mean_counts_24m.sum())),
      int(_dense_counts(simulated_hist_24m).sum()))
print("Three-month observed / injected expectation:",
      f"{_dense_counts(data_hist).sum() / (mean_counts_24m.sum() / EXPOSURE_SCALE_24M):.4f}")
del injection_folding_24m, source_expectation_3m, mean_counts_24m
gc.collect()

class ScaledExpectationPoissonLikelihood(PoissonLikelihood):
    """Fit a longer exposure with the unchanged three-month response cache."""
    def __init__(self, data, response, background, exposure_scale):
        super().__init__(data, response, background)
        self.exposure_scale = float(exposure_scale)
        if not np.isfinite(self.exposure_scale) or self.exposure_scale <= 0:
            raise ValueError("exposure_scale must be positive")

    def get_log_like(self):
        expectation = self._response.expectation(copy=True)
        expectation += self._bkg.expectation(copy=False)
        expected = self.exposure_scale * expectation.contents
        observed = self._data.data.contents
        # Same Poisson statistic as cosipy's PoissonLikelihood, but both sky
        # and instrumental-background expectations have the longer exposure.
        return float(np.nansum(observed * np.log(expected) - expected))


fit_model_24m = make_fit_model(deepcopy(parallel_model))
fit_background_24m = FreeNormBinnedBackground(
    {"total_bkg": total_bkg}, sc_history=sc_orientation, copy=True,
)
fit_folding_24m = BinnedThreeMLModelFolding(
    data=simulated_data_24m, point_source_response=psr,
    extended_source_response=esr,
    source_specific_point_source_responses=parallel_result["point responses for sed"],
)
fit_likelihood_24m = ScaledExpectationPoissonLikelihood(
    simulated_data_24m, fit_folding_24m, fit_background_24m,
    EXPOSURE_SCALE_24M,
)
fit_plugin_24m = ThreeMLPluginInterface(
    "cosi_24m_forecast", fit_likelihood_24m, fit_folding_24m,
    fit_background_24m,
)
fit_plugin_24m.bkg_parameter["total_bkg"] = Parameter(
    "total_bkg", background_rate_initial, min_value=0., max_value=50.,
    delta=max(0.05 * background_rate_initial, 1e-6), unit=u.Hz,
)
fit_joint_24m = JointLikelihood(
    fit_model_24m, DataList(fit_plugin_24m), verbose=False,
)
covariance_valid_24m = fit_joint_with_fallback(
    fit_joint_24m, "simulated 24-month full catalog",
)
best_shape_24m = (
    fit_joint_24m.results.optimized_model.ngc4151.spectrum.main.Cutoff_powerlaw
)
best_flux_24m = np.asarray(
    [best_shape_24m.evaluate_at(e) for e in energy], dtype=float,
)
low_24m = high_24m = None
if covariance_valid_24m:
    try:
        def cpl_flux_24m(e, K, index, xc):
            return K * (e / best_shape_24m.piv.value)**index * np.exp(-e / xc)
        propagated_24m = fit_joint_24m.results.propagate(
            cpl_flux_24m,
            K=fit_joint_24m.results.get_variates(best_shape_24m.K.path),
            index=fit_joint_24m.results.get_variates(best_shape_24m.index.path),
            xc=fit_joint_24m.results.get_variates(best_shape_24m.xc.path),
        )
        interval_24m = np.array([
            propagated_24m(e).equal_tail_interval(cl=0.68) for e in energy
        ])
        low_24m, high_24m = interval_24m[:, 0], interval_24m[:, 1]
    except (BadCovariance, CannotComputeCovariance, ValueError) as exc:
        print("24-month covariance band unavailable:", type(exc).__name__)

forecast_24m = {
    "K": float(best_shape_24m.K.value),
    "index": float(best_shape_24m.index.value),
    "xc (keV)": float(best_shape_24m.xc.value),
    "background (Hz)": float(fit_plugin_24m.bkg_parameter["total_bkg"].value),
    "fit flux": best_flux_24m,
    "fit energy keV": energy.copy(),
    "fit flux low 68": low_24m,
    "fit flux high 68": high_24m,
    "parameter intervals 68": parameter_intervals_68(
        fit_joint_24m.results,
        {"K": best_shape_24m.K, "index": best_shape_24m.index,
         "xc (keV)": best_shape_24m.xc,
         "background (Hz)": fit_plugin_24m.bkg_parameter["total_bkg"]},
        covariance_valid_24m,
    ),
}
print("24-month simulated full-catalog CPL fit (68% covariance intervals):")
for label, spec in (("K", ".4g"), ("index", ".3g"),
                    ("xc (keV)", ".4g"), ("background (Hz)", ".5g")):
    print(" ", label + ":", format_parameter_68(forecast_24m, label, spec))

# The independent SED fit uses the same native 10 Ei bins. Scale every fixed
# sky, background, and NGC template count by eight before fitting new counts.
forecast_24m["sed"] = extract_binned_sed(
    fit_model_24m, best_shape_24m,
    parallel_result["point responses for sed"], fit_background_24m,
    observed_hist=simulated_hist_24m, data_object=simulated_data_24m,
    ordinary_psr=psr, extended_psr=esr,
    label="simulated 24-month full catalog", counts_scale=EXPOSURE_SCALE_24M,
)

# Make a separate 24-month figure; the three-month comparison curves and SED
# points belong to different data and must not be relabeled as 24-month data.
sed_24m = forecast_24m["sed"]
pivot_24m = np.asarray(sed_24m["pivot keV"], dtype=float)
edges_24m = np.asarray(sed_24m["edges keV"], dtype=float)
detected_24m = (
    (sed_24m["TS"] >= 4.) & (sed_24m["K"] > 0.)
    & (sed_24m["lower 68"] > 0.) & np.isfinite(sed_24m["upper 68"])
)
upper_24m = (
    (~detected_24m) & np.isfinite(sed_24m["upper 95"])
    & (sed_24m["upper 95"] > 0.)
)
with paper_style("light", FONT_SIZE):
    figure_24m, axis_24m = plt.subplots(figsize=(12, 9), constrained_layout=True)
    style_axis(axis_24m, "light", FONT_SIZE)
    grid_24m = np.asarray(energy, dtype=float)
    sed_factor_24m = KEV_TO_ERG * grid_24m**2
    axis_24m.plot(
        grid_24m / 1000., sed_factor_24m * original_injected_flux(grid_24m),
        color=ink("light"), ls=":", lw=3, label="Injected total",
    )
    axis_24m.plot(
        grid_24m / 1000., sed_factor_24m * best_flux_24m,
        color=BLUE, lw=2.5,
        label="SC best fit & 68% band" if low_24m is not None else "SC best fit",
    )
    if low_24m is not None:
        axis_24m.fill_between(
            grid_24m / 1000., sed_factor_24m * low_24m,
            sed_factor_24m * high_24m, color=BLUE, alpha=0.27, linewidth=0,
        )
    if np.any(detected_24m):
        p = pivot_24m[detected_24m]
        k = sed_24m["K"][detected_24m]
        axis_24m.errorbar(
            p / 1000., KEV_TO_ERG * p**2 * k,
            xerr=np.vstack((p - edges_24m[:-1][detected_24m],
                            edges_24m[1:][detected_24m] - p)) / 1000.,
            yerr=np.vstack((KEV_TO_ERG * p**2 * (k - sed_24m["lower 68"][detected_24m]),
                            KEV_TO_ERG * p**2 * (sed_24m["upper 68"][detected_24m] - k))),
            fmt="o", color=BLUE, ecolor=BLUE, markerfacecolor="white",
            markeredgewidth=1.8, markersize=9, capsize=4,
            elinewidth=1.5, label="Forecast SED",
        )
    if np.any(upper_24m):
        p = pivot_24m[upper_24m]
        limit = KEV_TO_ERG * p**2 * sed_24m["upper 95"][upper_24m]
        axis_24m.errorbar(
            p / 1000., limit,
            xerr=np.vstack((p - edges_24m[:-1][upper_24m],
                            edges_24m[1:][upper_24m] - p)) / 1000.,
            yerr=np.maximum(0.5 * limit, np.finfo(float).tiny),
            uplims=True, fmt="v", color=BLUE, markerfacecolor="white",
            markeredgewidth=1.8, markersize=10, capsize=5, elinewidth=1.8,
        )
    axis_24m.set(
        xscale="log", yscale="log", xlim=(0.1, 10.),
        ylim=(1e-14, 2e-9), xlabel="Energy (MeV)",
        ylabel=r"Energy Flux (erg cm$^{-2}$ s$^{-1}$)",
    )
    axis_24m.xaxis.set_major_locator(mticker.FixedLocator([0.1, 1., 10.]))
    axis_24m.xaxis.set_major_formatter(
        mticker.FixedFormatter(["0.1", "1", "10"]),
    )
    axis_24m.text(
        0.97, 0.97, "NGC 4151\n24-month full-catalog "
        + ("Poisson" if FLUCTUATE_24M else "Asimov") + " forecast",
        transform=axis_24m.transAxes, ha="right", va="top",
        fontsize=FONT_SIZE, color=ink("light"),
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=5),
    )
    axis_24m.legend(loc="lower left", frameon=False, fontsize=FONT_SIZE)
    if SAVE_PLOTS:
        PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        stem_24m = ("NGC4151_DC4_paper_SC_24month_"
                    + ("poisson" if FLUCTUATE_24M else "asimov")
                    + "_forecast_white")
        figure_24m.savefig(
            PAPER_OUTPUT_DIR / f"{stem_24m}.pdf", format="pdf",
            facecolor="white", edgecolor="white", transparent=False,
            bbox_inches="tight",
        )
        figure_24m.savefig(
            PAPER_OUTPUT_DIR / f"{stem_24m}.png", dpi=220,
            facecolor="white", edgecolor="white", transparent=False,
            bbox_inches="tight",
        )
        print("Saved 24-month forecast plot to", PAPER_OUTPUT_DIR)
    plt.show()
